# Eksperimen Terjemahan Mesin Batak Toba dan Bahasa Indonesia pada Korpus Paralel Multi-Genre Terverifikasi Varietas

**Notebook eksperimen untuk naskah jurnal tahap 2**
Penelitian Dosen Pemula UPH: Korpus Digital Batak Toba dan Bahasa Indonesia

Notebook ini menjalankan seluruh eksperimen naskah berikutnya, mulai dari memuat data sampai menyiapkan tabel untuk naskah. Setiap tahap menyimpan hasilnya ke Google Drive sebagai *checkpoint* (titik simpan). Kalau *runtime* (sesi Colab) terputus atau *compute unit* (satuan komputasi) habis, Anda cukup membuka notebook lagi lalu menjalankannya ulang. Tahap yang sudah selesai akan dilewati otomatis, dan tahap yang terhenti di tengah jalan akan dilanjutkan dari titik terakhir.

| Bagian | Isi |
|---|---|
| 0 | Latar belakang, rumusan masalah, tujuan, tinjauan literatur, peta tahapan |
| A | Persiapan: GPU, Google Drive, pustaka, konfigurasi, utilitas *checkpoint* |
| B | Data: korpus kita, NusaX, NusaTranslation, audit varietas, pembagian data |
| C | Eksperimen model: *baseline*, *zero-shot*, *fine-tuning*, LLM *few-shot* |
| D | Evaluasi otomatis dan analisis per arah penerjemahan asli |
| E | Evaluasi manusia (ESA) dengan dua penilai |
| F | Paket hasil untuk naskah |

## Cara memakai notebook ini

1. **Pilih GPU.** Menu *Runtime* > *Change runtime type* > pilih **A100** (disarankan) atau **T4**. Bagian B tidak butuh GPU, tetapi bagian C butuh.
2. **Jalankan sel dari atas ke bawah**, atau pakai *Runtime* > *Run all*. Sel konfigurasi (A.2) adalah satu-satunya sel yang biasanya perlu Anda ubah.
3. **Kalau terputus:** sambungkan lagi, lalu pilih *Run all*. Tahap yang sudah selesai hanya mencetak ringkasan dari *checkpoint*. Pelatihan model dilanjutkan dari langkah terakhir yang tersimpan, dan penerjemahan dilanjutkan dari kalimat terakhir yang tersimpan.
4. **Kalau ingin mengulang satu tahap**, tulis kodenya di `paksa_ulang` pada sel konfigurasi, misalnya `["S04"]`.

**Berkas yang perlu Anda siapkan di Google Drive**, di folder `MyDrive/Datasets/btb-eksperimen-mt/input/`:

| Berkas | Wajib? | Keterangan |
|---|---|---|
| `arah_penerjemahan_periksa.xlsx` | Sangat disarankan | Lembar label arah penerjemahan yang sudah Anda periksa (kolom `arah_final`). Kalau belum ada, notebook memakai label otomatis dari GitHub dan memberi peringatan. |
| `btb_parallel_sentence.csv` | Tidak | Kalau tidak ada, notebook mengunduhnya dari repositori GitHub. |

NusaX dan NusaTranslation diunduh otomatis dari repositori IndoNLP di GitHub.

**Token Hugging Face** (hanya perlu bila model yang dipakai bersifat *gated* atau terkunci): simpan di menu kunci (*Secrets*) Colab dengan nama `HF_TOKEN`. **Jangan pernah menulis token langsung di sel kode**, karena notebook ini akan disimpan di GitHub.

---
# Bagian 0. Landasan penelitian

## 0.1 Latar belakang masalah

Bahasa Batak Toba (kode ISO 639-3: `bbc`) dituturkan oleh sekitar dua juta orang di Sumatera Utara dan di perantauan, tetapi sumber daya digitalnya untuk *machine translation* (MT, terjemahan mesin) masih sangat terbatas. Model multibahasa besar seperti NLLB-200 (NLLB Team dkk., 2022) tidak mencakup Batak Toba. Kode `bbc_Latn` tidak ada di daftar 202 bahasanya, sehingga peneliti harus meminjam kode bahasa lain atau menambah kode baru.

Data paralel Batak Toba dan Bahasa Indonesia yang tersedia secara terbuka saat ini terutama berasal dari dua korpus:

1. **NusaX** (Winata dkk., 2023): 1.000 kalimat berlabel *Toba Batak* (`bbc`), diterjemahkan dari dataset sentimen berbahasa Indonesia (ulasan dan komentar).
2. **NusaTranslation** (Cahyawijaya dkk., 2023): 9.449 kalimat berlabel `btk`, juga diterjemahkan dari Bahasa Indonesia (ulasan hotel dan cuitan). Dalam ISO 639-2, `btk` adalah kode **rumpun** bahasa Batak (Toba, Angkola, Mandailing, Karo, Simalungun, Pakpak), bukan kode Batak Toba.

Naskah kami sebelumnya yang sedang dalam proses review menunjukkan bahwa sekitar sepertiga rujukan uji NusaTranslation bukan Batak Toba, sebagian besar Angkola-Mandailing. Model yang dilatih dengan data ini justru **memperkuat** varietas tersebut pada keluarannya, dan metrik berbasis rujukan memberi nilai pada kecocokan varietas, bukan pada mutu terjemahan Batak Toba. Kesimpulannya: dibutuhkan data uji Batak Toba yang **identitas varietasnya terdokumentasi** dan, idealnya, **berasal dari teks yang aslinya ditulis dalam Batak Toba**, bukan hanya terjemahan dari Bahasa Indonesia.

Masalah kedua adalah ***translationese*** (ciri khas teks terjemahan). Teks terjemahan cenderung lebih sederhana, lebih harfiah, dan lebih dekat ke bahasa sumbernya. Penelitian pada pasangan bahasa besar menunjukkan bahwa arah penerjemahan data uji dapat mengubah skor dan bahkan peringkat sistem (Zhang dan Toral, 2019; Graham dkk., 2020). Untuk Batak Toba, seluruh data uji publik adalah terjemahan dari Bahasa Indonesia, sehingga kita belum tahu seberapa baik model menerjemahkan teks Batak Toba yang ditulis secara alami.

Korpus paralel multi-genre kita (1.542 pasangan kalimat dari 97 dokumen) menjawab kedua masalah ini. Korpus ini dikurasi dan diterjemahkan oleh penutur Batak Toba, mencakup lima genre (media kontemporer, edukasi, sastra, sejarah dan tradisi, religi), memiliki laporan kesepakatan antaranotator, dan sebagian dokumennya aslinya ditulis dalam Batak Toba. Uji awal dengan daftar penanda leksikal dari naskah sebelumnya menunjukkan bahwa hanya 0,3% kalimat Batak kita yang bertanda Angkola-Mandailing, dibandingkan 24,6% pada rujukan uji NusaTranslation.

## 0.2 Rumusan masalah

1. **RQ1 (mutu data).** Apakah korpus paralel multi-genre ini dapat diandalkan sebagai sumber daya Batak Toba, dilihat dari kesepakatan antaranotator dan dari audit varietas?
2. **RQ2 (arah penerjemahan).** Apakah kinerja sistem MT berbeda antara teks yang aslinya berbahasa Batak Toba dan teks yang aslinya berbahasa Indonesia?
3. **RQ3 (kemampuan model yang ada).** Seberapa baik model terjemahan multibahasa (NLLB-200, MADLAD-400) dan *large language model* (LLM, model bahasa besar) menerjemahkan Batak Toba tanpa pelatihan tambahan?
4. **RQ4 (nilai data latih).** Apakah menambahkan korpus kita ke data latih (NusaX atau NusaTranslation) meningkatkan kinerja, terutama pada data uji di luar domain data latih?
5. **RQ5 (contoh dalam *prompt*).** Apakah contoh terjemahan yang diambil dari korpus kita berdasarkan kemiripan (*retrieval*) membantu LLM lebih baik daripada contoh acak?
6. **RQ6 (validitas evaluasi).** Seberapa sejalan metrik otomatis dengan penilaian dua penutur asli Batak Toba?

## 0.3 Tujuan penelitian

1. Menyusun pembagian data latih, validasi, dan uji berbasis dokumen yang dapat direproduksi, lengkap dengan label genre dan arah penerjemahan asli.
2. Mengukur kinerja *baseline* dan model *zero-shot* (tanpa pelatihan tambahan) pada tiga set uji: korpus kita, NusaX, dan NusaTranslation.
3. Mengukur pengaruh data latih dari korpus kita melalui *fine-tuning* (pelatihan lanjutan) NLLB-200.
4. Mengukur pengaruh *in-context learning* (pembelajaran dari contoh dalam *prompt*) dengan contoh acak dan contoh hasil *retrieval* pada LLM terbuka.
5. Membandingkan skor otomatis dengan *Error Span Annotation* (ESA, anotasi rentang kesalahan) dari dua penilai, termasuk kesepakatan antarpenilai.

## 0.4 Tinjauan literatur

**MT untuk bahasa daerah Indonesia.** Indonesia memiliki lebih dari 700 bahasa, tetapi sebagian besar hampir tidak punya data digital (Aji dkk., 2022). NusaX (Winata dkk., 2023) dan NusaWrites/NusaTranslation (Cahyawijaya dkk., 2023) adalah upaya terbesar untuk menyediakan data paralel bahasa daerah, dan keduanya kini tersedia melalui SEACrowd (Lovenia dkk., 2024). Kedua korpus dibuat dengan menerjemahkan teks Indonesia ke bahasa daerah. Untuk Batak Toba, Samuel dan Ali (2024) melakukan *fine-tuning* NLLB-200 pada data `btk` NusaTranslation dan melaporkannya sebagai sistem Batak Toba.

**Label bahasa dan varietas.** Kreutzer dkk. (2022) menemukan banyak korpus multibahasa hasil perayapan web yang salah label atau memakai kode bahasa yang ambigu. FRMT (Riley dkk., 2023) memperkenalkan evaluasi MT yang peka varietas regional. Ali dkk. (2026) menunjukkan bahwa mengganti rujukan FLORES+ dengan rujukan dari varietas yang berkerabat dekat mengubah skor MT bahasa Bantu di Mozambik secara besar. Naskah kami sebelumnya menunjukkan masalah serupa yang tersembunyi di dalam satu set uji Batak berlabel `btk`.

**Arah penerjemahan dan *translationese*.** Zhang dan Toral (2019) serta Graham dkk. (2020) menunjukkan bahwa data uji yang sisi sumbernya adalah teks terjemahan dapat menggelembungkan skor dan mengubah kesimpulan evaluasi. Freitag dkk. (2020) menunjukkan bahwa rujukan yang bergaya *translationese* membuat BLEU kurang andal. Studi-studi ini dilakukan pada bahasa berdaya sumber tinggi, dan belum ada yang menelitinya untuk Batak Toba.

**Menambahkan bahasa baru ke model multibahasa.** Bila bahasa sasaran tidak punya kode bahasa di model, peneliti dapat meminjam kode bahasa yang berkerabat atau menambah *token* bahasa baru yang diinisialisasi dari rata-rata *embedding* beberapa bahasa (Ntohsi dkk., 2026). Naskah kami sebelumnya menemukan bahwa kode Banjar (`bjn_Latn`) sedikit lebih baik daripada token baru untuk arah Batak ke Indonesia. Sennrich dan Zhang (2019) menunjukkan bahwa hasil MT pada data kecil sangat bergantung pada pengaturan pelatihan yang sesuai dengan ukuran data. MADLAD-400 (Kudugunta dkk., 2023) adalah model MT yang dilatih pada data web lebih dari 400 bahasa dan mungkin mencakup Batak Toba. Notebook ini memeriksanya secara otomatis.

**LLM untuk MT bahasa berdaya sumber rendah.** LLM mampu menerjemahkan bahasa besar dengan baik, tetapi kinerjanya menurun tajam untuk bahasa berdaya sumber rendah (Hendy dkk., 2023; Robinson dkk., 2023). Pemilihan contoh dalam *prompt* berpengaruh besar: contoh yang mirip dengan kalimat yang diterjemahkan umumnya lebih membantu daripada contoh acak (Vilar dkk., 2023; Agrawal dkk., 2023).

**Evaluasi MT.** chrF++ (Popović, 2017) bekerja pada *n-gram* karakter dan kata sehingga lebih tahan terhadap variasi ejaan, dan disarankan untuk bahasa yang tidak didukung metrik terlatih (Kocmi dkk., 2021). BLEU (Papineni dkk., 2002) dilaporkan dengan sacreBLEU (Post, 2018) agar dapat dibandingkan. COMET (Rei dkk., 2022) dibangun di atas XLM-R yang tidak mencakup bahasa Batak, sehingga hasilnya untuk Batak tidak andal (Zouhar dkk., 2024). Di sini COMET hanya dipakai sebagai bahan analisis. Untuk evaluasi manusia, ESA (Kocmi dkk., 2024) meminta penilai menandai rentang kesalahan lalu memberi skor 0 sampai 100. Perbedaan antarsistem diuji dengan *paired bootstrap resampling* (Koehn, 2004).

**Celah yang diisi penelitian ini.** Belum ada data uji Batak Toba yang (1) terverifikasi varietasnya, (2) mencakup beberapa genre, dan (3) mencatat arah penerjemahan asli setiap dokumen. Belum ada pula studi yang membandingkan *fine-tuning* NLLB, MADLAD-400, dan LLM *few-shot* untuk Batak Toba pada data seperti itu.

### Daftar pustaka

- Agrawal, S., Zhou, C., Lewis, M., Zettlemoyer, L., dan Ghazvininejad, M. (2023). In-context examples selection for machine translation. *Findings of ACL 2023*.
- Aji, A. F., Winata, G. I., Koto, F., dkk. (2022). One country, 700+ languages: NLP challenges for underrepresented languages and dialects in Indonesia. *ACL 2022*.
- Ali, F. D. M. A., Mateus, D. L., dan Mangue, M. V. (2026). Measuring the cost of variety conflation in multilingual MT evaluation: Adding Mozambican Xichangana, Nyanja and Sena to FLORES+. *arXiv:2609.13847*.
- Cahyawijaya, S., Lovenia, H., Koto, F., dkk. (2023). NusaWrites: Constructing high-quality corpora for underrepresented and extremely low-resource languages. *IJCNLP-AACL 2023*.
- Freitag, M., Grangier, D., dan Caswell, I. (2020). BLEU might be guilty but references are not innocent. *EMNLP 2020*.
- Graham, Y., Haddow, B., dan Koehn, P. (2020). Statistical power and translationese in machine translation evaluation. *EMNLP 2020*.
- Hendy, A., Abdelrehim, M., Sharaf, A., dkk. (2023). How good are GPT models at machine translation? A comprehensive evaluation. *arXiv:2302.09210*.
- Kocmi, T., Federmann, C., Grundkiewicz, R., dkk. (2021). To ship or not to ship: An extensive evaluation of automatic metrics for machine translation. *WMT 2021*.
- Kocmi, T., Zouhar, V., Avramidis, E., dkk. (2024). Error span annotation: A balanced approach for human evaluation of machine translation. *WMT 2024*.
- Koehn, P. (2004). Statistical significance tests for machine translation evaluation. *EMNLP 2004*.
- Kreutzer, J., Caswell, I., Wang, L., dkk. (2022). Quality at a glance: An audit of web-crawled multilingual datasets. *TACL*, 10.
- Kudugunta, S., Caswell, I., Zhang, B., dkk. (2023). MADLAD-400: A multilingual and document-level large audited dataset. *NeurIPS 2023 Datasets and Benchmarks*.
- Lovenia, H., Mahendra, R., Akbar, S. M., dkk. (2024). SEACrowd: A multilingual multimodal data hub and benchmark suite for Southeast Asian languages. *EMNLP 2024*.
- NLLB Team, Costa-jussà, M. R., Cross, J., dkk. (2022). No language left behind: Scaling human-centered machine translation. *arXiv:2207.04672*.
- Ntohsi, S., Tuyishimire, N. D., Kafesu, A., dkk. (2026). Embedding initialization for unseen low-resource languages in multilingual NMT: A case study on Limbum-English translation. *arXiv:2608.07629*.
- Papineni, K., Roukos, S., Ward, T., dan Zhu, W.-J. (2002). BLEU: A method for automatic evaluation of machine translation. *ACL 2002*.
- Popović, M. (2017). chrF++: Words helping character n-grams. *WMT 2017*.
- Post, M. (2018). A call for clarity in reporting BLEU scores. *WMT 2018*.
- Rei, R., de Souza, J. G. C., Alves, D., dkk. (2022). COMET-22: Unbabel-IST 2022 submission for the metrics shared task. *WMT 2022*.
- Riley, P., Dozat, T., Botha, J. A., dkk. (2023). FRMT: A benchmark for few-shot region-aware machine translation. *TACL*, 11.
- Robinson, N. R., Ogayo, P., Mortensen, D. R., dan Neubig, G. (2023). ChatGPT MT: Competitive for high- (but not low-) resource languages. *WMT 2023*.
- Samuel, C., dan Ali, I. T. (2024). Batak Toba language-Indonesian machine translation with transfer learning using no language left behind. *International Journal of Advances in Applied Sciences*, 13(4), 834-843.
- Sennrich, R., dan Zhang, B. (2019). Revisiting low-resource neural machine translation: A case study. *ACL 2019*.
- Vilar, D., Freitag, M., Cherry, C., dkk. (2023). Prompting PaLM for translation: Assessing strategies and performance. *ACL 2023*.
- Winata, G. I., Aji, A. F., Cahyawijaya, S., dkk. (2023). NusaX: Multilingual parallel sentiment dataset for 10 Indonesian local languages. *EACL 2023*.
- Zhang, M., dan Toral, A. (2019). The effect of translationese in machine translation test sets. *WMT 2019*.
- Zouhar, V., Chen, P., Lam, T. K., dkk. (2024). Pitfalls and outlooks in using COMET. *WMT 2024*.

> Catatan: periksa ulang setiap rujukan ke sumber aslinya sebelum dipakai di naskah. Rujukan ke naskah kami yang sedang direview sengaja tidak menyebut judul. Sesuaikan dengan aturan anonimitas venue.

## 0.5 Peta tahapan dan *checkpoint*

| Kode | Tahap | RQ | GPU? | Perkiraan waktu (A100) | *Checkpoint* yang disimpan |
|---|---|---|---|---|---|
| S01 | Muat korpus kita dan label arah | RQ1, RQ2 | Tidak | < 1 menit | `keluaran/s01_korpus.csv` |
| S02 | Unduh NusaX dan NusaTranslation | RQ3, RQ4 | Tidak | < 1 menit | `keluaran/s02_*.csv` |
| S03 | Audit varietas (penanda leksikal) | RQ1 | Tidak | < 1 menit | `keluaran/s03_*.csv`, gambar |
| S04 | Pembagian data berbasis dokumen dan cek tumpang tindih | semua | Tidak | < 1 menit | `keluaran/s04_*.csv` |
| S05 | Susun set uji dan konfigurasi data latih | semua | Tidak | < 1 menit | `keluaran/s05_*.csv` |
| S06 | *Baseline copy-source* dan NLLB *zero-shot* | RQ3 | Ya | 20 sampai 40 menit | `prediksi/` per kalimat |
| S07 | MADLAD-400 *zero-shot* (bila mencakup Batak Toba) | RQ3 | Ya | 15 sampai 30 menit | `prediksi/` per kalimat |
| S08 | *Fine-tuning* NLLB-200 pada beberapa konfigurasi data | RQ4 | Ya | 1 sampai 2 jam | bobot model per langkah, `prediksi/` |
| S09 | LLM *zero-shot* dan *few-shot* | RQ3, RQ5 | Ya | 1 sampai 2 jam | `prediksi/` per kalimat |
| S10 | Evaluasi otomatis semua sistem | semua | Tidak | 5 sampai 10 menit | `keluaran/s10_*.csv` |
| S11 | Analisis per arah asli, genre, dan varietas keluaran | RQ2 | Tidak | < 1 menit | `keluaran/s11_*.csv`, gambar |
| S12 | Siapkan lembar ESA untuk dua penilai | RQ6 | Tidak | < 1 menit | `evaluasi_manusia/` |
| S13 | Analisis ESA dan kesepakatan antarpenilai | RQ6 | Tidak | < 1 menit | `keluaran/s13_*.csv` |
| S14 | Paket hasil untuk naskah | semua | Tidak | < 1 menit | `rilis/` |

Ada dua jenis *checkpoint*:

1. **Tingkat tahap.** Setiap tahap mencatat *fingerprint* (sidik) berupa parameter, *hash* SHA-256 berkas masukan, dan *hash* kode selnya di `keluaran/_meta/`. Kalau ketiganya tidak berubah dan berkas keluarannya ada, tahap dilewati.
2. **Tingkat unit kerja** (S06 sampai S09). Setiap terjemahan kalimat langsung ditulis ke berkas `.jsonl` di Drive, dan pelatihan model menyimpan bobot serta *optimizer state* (keadaan pengoptimal) setiap beberapa ratus langkah. Pekerjaan yang terhenti dilanjutkan dari titik itu.

---
# Bagian A. Persiapan

## A.1 Cek GPU, sambungkan Google Drive, dan pasang pustaka

**Yang dilakukan sel ini:** memeriksa jenis GPU, menyambungkan Google Drive, lalu memasang pustaka yang belum ada di Colab (`sacrebleu` untuk metrik, `sentencepiece` untuk *tokenizer*, `openpyxl` untuk berkas Excel).

**Keluaran yang diharapkan:** nama GPU (misalnya `NVIDIA A100-SXM4-40GB`), pesan `Mounted at /content/drive`, dan versi pustaka utama. Kalau muncul `GPU: tidak ada`, bagian B tetap bisa dijalankan, tetapi bagian C tidak.

In [ ]:
# A.1 Persiapan lingkungan
import subprocess, sys, os

# 1) Cek GPU. nvidia-smi hanya ada bila runtime memakai GPU.
try:
    info_gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                              capture_output=True, text=True, check=True).stdout.strip()
    print("GPU:", info_gpu)
except Exception:
    info_gpu = ""
    print("GPU: tidak ada. Bagian B bisa jalan, bagian C butuh GPU.")

# 2) Sambungkan Google Drive agar semua checkpoint tersimpan permanen.
DI_COLAB = "google.colab" in sys.modules
if DI_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# 3) Pasang pustaka yang belum tersedia. Pustaka besar (torch, transformers) sudah ada di Colab.
def pasang_bila_perlu(modul, paket):
    try:
        __import__(modul)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paket], check=True)

for modul, paket in [("sacrebleu", "sacrebleu>=2.4"), ("sentencepiece", "sentencepiece"),
                     ("openpyxl", "openpyxl"), ("sklearn", "scikit-learn"), ("accelerate", "accelerate")]:
    pasang_bila_perlu(modul, paket)

import sacrebleu, pandas, numpy
print("sacrebleu", sacrebleu.__version__, "| pandas", pandas.__version__, "| numpy", numpy.__version__)
try:
    import torch, transformers
    print("torch", torch.__version__, "| transformers", transformers.__version__,
          "| CUDA:", torch.cuda.is_available())
except ImportError:
    print("torch atau transformers belum terpasang.")

## A.2 Konfigurasi (satu-satunya sel yang biasanya perlu diubah)

**Yang dilakukan sel ini:** menetapkan lokasi folder kerja, sumber data, model, *hyperparameter* (parameter pelatihan), dan pilihan eksperimen. Semua tahap membaca nilainya dari sini.

**Hal yang paling sering diubah:**

- `paksa_ulang`: daftar kode tahap yang ingin dijalankan ulang walaupun sudah ada *checkpoint*-nya.
- `nllb_zero_shot`, `madlad`, `llm.model`: daftar model. Hapus model yang tidak ingin dijalankan agar hemat *compute unit*.
- `fine_tuning.konfigurasi_data` dan `fine_tuning.benih`: kombinasi data latih dan *random seed* (benih acak).
- `fine_tuning.ukuran_batch` dan `akumulasi_gradien`: untuk T4, pakai `8` dan `2`; untuk A100, `16` dan `1`. Hasil kalinya (16) sebaiknya tetap.

**Keluaran yang diharapkan:** daftar folder yang dibuat di Drive.

In [ ]:
# A.2 Konfigurasi
from pathlib import Path

KONFIGURASI = {
    "versi_pipeline": "1.0.0",
    # Folder kerja di Google Drive. Semua checkpoint dan hasil disimpan di sini.
    "folder_kerja": "/content/drive/MyDrive/Datasets/btb-eksperimen-mt",
    # Tahap yang dipaksa dijalankan ulang, misalnya ["S04", "S10"].
    "paksa_ulang": [],
    "benih_global": 42,

    # ---------- Sumber data ----------
    "url_korpus": "https://raw.githubusercontent.com/FeliksMakarios/batak-toba-korpus/main/data/btb_parallel_sentence.csv",
    "url_label_arah_otomatis": "https://raw.githubusercontent.com/FeliksMakarios/batak-toba-korpus/main/metadata/arah_penerjemahan.csv",
    "berkas_label_arah_terperiksa": "arah_penerjemahan_periksa.xlsx",   # di folder input/
    "url_nusax": "https://raw.githubusercontent.com/IndoNLP/nusax/main/datasets/mt/{split}.csv",
    "kolom_nusax_batak": "toba_batak",
    "url_nusatranslation": "https://raw.githubusercontent.com/IndoNLP/nusa-writes/main/data/nusa_kalimat-mt-btk-{split}.csv",

    # ---------- Pembagian data korpus kita (S04) ----------
    "pembagian": {"porsi_uji": 0.20, "porsi_validasi": 0.10, "toleransi_lebih": 1.25},
    # Ambang kemiripan Jaccard token untuk dianggap hampir sama (cek tumpang tindih).
    "ambang_jaccard_duplikat": 0.9,

    # ---------- Kode bahasa ----------
    "kode_nllb_indonesia": "ind_Latn",
    "kode_nllb_batak": "bjn_Latn",      # NLLB tidak punya bbc_Latn; Banjar dipinjam (lihat 0.4)
    "kode_madlad_indonesia": "<2id>",
    "kode_madlad_batak": "<2bbc>",

    # ---------- S06: zero-shot NLLB ----------
    "nllb_zero_shot": ["facebook/nllb-200-distilled-600M",
                       "facebook/nllb-200-distilled-1.3B",
                       "facebook/nllb-200-3.3B"],
    "set_uji_zero_shot": ["kami", "nusax", "nusatranslation"],
    "dekode": {"beam": 5, "ukuran_batch": 32, "panjang_maks_sumber": 128,
               "faktor_panjang_keluaran": 1.5, "tambahan_panjang": 10, "panjang_maks_keluaran": 256},

    # ---------- S07: MADLAD-400 ----------
    "madlad": ["google/madlad400-3b-mt"],

    # ---------- S08: fine-tuning NLLB ----------
    "fine_tuning": {
        "model_dasar": "facebook/nllb-200-distilled-600M",
        # Nama konfigurasi data latih. "kami" = data latih korpus kita (S04).
        "konfigurasi_data": ["nusax", "nusatranslation", "kami", "kami+nusax", "kami+nusatranslation"],
        # Benih per konfigurasi. Konfigurasi kecil diulang tiga kali karena variansnya lebih besar.
        "benih": {"nusax": [42, 1, 2], "kami": [42, 1, 2], "kami+nusax": [42, 1, 2],
                  "nusatranslation": [42], "kami+nusatranslation": [42]},
        "dua_arah": True,            # satu model untuk id->bbc dan bbc->id
        "epoch": 5,
        "laju_belajar": 1e-4,
        "porsi_warmup": 0.10,
        "ukuran_batch": 16,          # T4: 8
        "akumulasi_gradien": 1,      # T4: 2
        "panjang_maks": 128,
        "simpan_tiap_langkah": 300,  # checkpoint bobot model ke Drive setiap N langkah optimizer
        "hapus_bobot_setelah_selesai": True,  # hemat ruang Drive setelah semua prediksi tersimpan
    },

    # ---------- S09: LLM ----------
    "llm": {
        # Periksa nama dan lisensi setiap model di Hugging Face sebelum menjalankan.
        "model": ["Qwen/Qwen2.5-7B-Instruct",
                  "GoToCompany/gemma2-9b-cpt-sahabatai-v1-instruct"],
        "kondisi": ["zero_shot", "few_shot_acak", "few_shot_retrieval"],
        "k_contoh": 5,
        "set_uji": ["kami", "nusax", "nusatranslation_toba"],
        "sampel_nusatranslation_toba": 400,   # subsampel agar hemat waktu
        "ukuran_batch": 16,
        "token_baru_maks": 200,
    },

    # ---------- S10: evaluasi otomatis ----------
    "evaluasi": {"n_bootstrap": 1000, "hitung_comet": False, "model_comet": "Unbabel/wmt22-comet-da"},

    # ---------- S12: evaluasi manusia ----------
    "esa": {
        "n_kalimat": 60,             # kalimat dari set uji korpus kita, berstrata arah asli
        # Sistem yang dinilai. "LLM_TERBAIK" diganti otomatis dengan kondisi LLM ber-chrF++ tertinggi.
        "sistem": ["nllb600m_zeroshot", "ft_nusatranslation_s42", "ft_kami+nusatranslation_s42", "LLM_TERBAIK"],
        "penilai": ["A", "B"],
    },
}

# ---------- Struktur folder ----------
DIR_KERJA = Path(KONFIGURASI["folder_kerja"])
DIR_INPUT = DIR_KERJA / "input"
DIR_KELUARAN = DIR_KERJA / "keluaran"
DIR_META = DIR_KELUARAN / "_meta"
DIR_PREDIKSI = DIR_KERJA / "prediksi"
DIR_MODEL = DIR_KERJA / "model_sementara"
DIR_ESA = DIR_KERJA / "evaluasi_manusia"
DIR_GAMBAR = DIR_KELUARAN / "gambar"
DIR_RILIS = DIR_KERJA / "rilis"
for d in [DIR_INPUT, DIR_KELUARAN, DIR_META, DIR_PREDIKSI, DIR_MODEL, DIR_ESA / "terisi", DIR_GAMBAR, DIR_RILIS]:
    d.mkdir(parents=True, exist_ok=True)
print("Folder kerja:", DIR_KERJA)
for d in sorted(p for p in DIR_KERJA.iterdir() if p.is_dir()):
    print("  ", d.name + "/")

## A.3 Utilitas: *checkpoint*, berkas, dan tampilan

**Yang dilakukan sel ini:** mendefinisikan fungsi bantu yang dipakai semua tahap.

- `Tahap`: pencatat *checkpoint* tingkat tahap. Di awal tahap, `Tahap(...).bisa_dilewati()` memeriksa apakah tahap ini sudah pernah selesai dengan masukan, parameter, dan kode yang sama. Di akhir tahap, `.selesai(ringkasan)` menyimpan catatannya.
- `tulis_atomik`: menulis berkas lewat berkas sementara lalu mengganti namanya, supaya berkas di Drive tidak rusak kalau sesi terputus saat menulis.
- `UnitPrediksi`: berkas `.jsonl` berisi terjemahan per kalimat. Kalimat yang sudah diterjemahkan tidak diterjemahkan ulang, dan parameter yang berubah memicu terjemahan ulang.
- Penanda varietas Angkola-Mandailing (AM) dan Toba dari naskah kami sebelumnya (32 dan 12 kata).

**Keluaran yang diharapkan:** pesan `Utilitas siap.`

In [ ]:
# A.3 Utilitas umum
import hashlib, json, random, re, time, shutil, math, gc
from datetime import datetime, timezone
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)
STATUS_TAHAP = {}


def sha256_berkas(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for blok in iter(lambda: f.read(1 << 20), b""):
            h.update(blok)
    return h.hexdigest()


def sidik_objek(obj):
    teks = json.dumps(obj, sort_keys=True, ensure_ascii=False, default=str)
    return hashlib.sha256(teks.encode("utf-8")).hexdigest()


def sumber_sel_aktif():
    # Teks sel yang sedang dijalankan; dipakai agar perubahan kode memicu tahap dijalankan ulang.
    try:
        return get_ipython().user_ns["_ih"][-1]
    except Exception:
        return ""


def tulis_atomik(path, isi_teks):
    path = Path(path)
    sementara = path.with_suffix(path.suffix + ".tmp")
    sementara.write_text(isi_teks, encoding="utf-8")
    os.replace(sementara, path)


def simpan_csv(df, path):
    path = Path(path)
    sementara = path.with_suffix(".tmp.csv")
    df.to_csv(sementara, index=False, encoding="utf-8")
    os.replace(sementara, path)


def baca_csv(path):
    return pd.read_csv(path, keep_default_na=False, na_values=[""], dtype=str)


def tampilkan(df, n=10):
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())


class Tahap:
    # Checkpoint tingkat tahap berdasarkan sidik (parameter + hash masukan + hash kode sel).
    def __init__(self, kode, nama, masukan, keluaran, parameter):
        self.kode, self.nama = kode, nama
        self.masukan = [Path(p) for p in masukan]
        self.keluaran = [Path(p) for p in keluaran]
        self.parameter = parameter
        hilang = [str(p) for p in self.masukan if not p.exists()]
        if hilang:
            raise FileNotFoundError(f"[{kode}] Berkas masukan belum ada: {hilang}. Jalankan tahap sebelumnya dulu.")
        self.sidik_masukan = {p.name: sha256_berkas(p) for p in self.masukan}
        self.sidik = sidik_objek({"versi": KONFIGURASI["versi_pipeline"], "parameter": parameter,
                                  "masukan": self.sidik_masukan, "kode_sel": sidik_objek(sumber_sel_aktif())})
        self.path_meta = DIR_META / f"{kode}.json"

    def bisa_dilewati(self):
        if self.kode in KONFIGURASI["paksa_ulang"]:
            return False
        if not self.path_meta.exists() or not all(p.exists() for p in self.keluaran):
            return False
        meta = json.loads(self.path_meta.read_text(encoding="utf-8"))
        if meta.get("sidik") != self.sidik:
            return False
        STATUS_TAHAP[self.kode] = "checkpoint"
        print(f"[{self.kode}] {self.nama}: diambil dari checkpoint ({meta['waktu']}).")
        for k, v in meta.get("ringkasan", {}).items():
            print(f"    {k}: {v}")
        return True

    def selesai(self, ringkasan):
        meta = {"kode": self.kode, "nama": self.nama, "sidik": self.sidik,
                "waktu": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                "parameter": self.parameter, "masukan": self.sidik_masukan,
                "keluaran": {p.name: sha256_berkas(p) for p in self.keluaran if p.is_file()},
                "ringkasan": ringkasan}
        tulis_atomik(self.path_meta, json.dumps(meta, indent=2, ensure_ascii=False, default=str))
        STATUS_TAHAP[self.kode] = "dijalankan"
        print(f"[{self.kode}] {self.nama}: selesai dan disimpan.")
        for k, v in ringkasan.items():
            print(f"    {k}: {v}")


class UnitPrediksi:
    # Berkas prediksi per (sistem, set uji, arah). Baris pertama berisi parameter; baris lain satu kalimat.
    def __init__(self, sistem, set_uji, arah, parameter, buat=True):
        # buat=False: hanya memeriksa kelengkapan, tanpa membuat atau memindahkan berkas.
        self.sistem, self.set_uji, self.arah = sistem, set_uji, arah
        self.parameter = parameter
        self.path = DIR_PREDIKSI / sistem / f"{set_uji}.{arah}.jsonl"
        if buat:
            self.path.parent.mkdir(parents=True, exist_ok=True)
        self.sidik = sidik_objek(parameter)
        self.selesai_ids = {}
        if self.path.exists():
            baris = self.path.read_text(encoding="utf-8").splitlines()
            kepala = json.loads(baris[0]) if baris else {}
            if kepala.get("sidik") == self.sidik:
                for b in baris[1:]:
                    try:
                        r = json.loads(b)
                        self.selesai_ids[r["uji_id"]] = r["hipotesis"]
                    except Exception:
                        pass  # baris terakhir yang terpotong saat sesi putus diabaikan
            elif buat:
                print(f"  Parameter {self.path.name} berubah; prediksi lama dipindahkan dan dibuat ulang.")
                os.replace(self.path, self.path.with_suffix(".lama.jsonl"))
        if buat and not self.path.exists():
            tulis_atomik(self.path, json.dumps({"sidik": self.sidik, "parameter": parameter}, default=str) + "\n")

    def sisa(self, df_uji):
        return df_uji[~df_uji["uji_id"].isin(self.selesai_ids)]

    def tambah(self, uji_ids, hipotesis):
        with open(self.path, "a", encoding="utf-8") as f:
            for i, h in zip(uji_ids, hipotesis):
                f.write(json.dumps({"uji_id": i, "hipotesis": h}, ensure_ascii=False) + "\n")
                self.selesai_ids[i] = h
            f.flush()
            os.fsync(f.fileno())

    def lengkap(self, df_uji):
        return len(self.sisa(df_uji)) == 0


def atur_benih(benih):
    random.seed(benih)
    np.random.seed(benih)
    try:
        import torch
        torch.manual_seed(benih)
        torch.cuda.manual_seed_all(benih)
    except ImportError:
        pass


def bersihkan_gpu():
    gc.collect()
    try:
        import torch
        torch.cuda.empty_cache()
    except ImportError:
        pass


# Penanda varietas dari naskah kami sebelumnya (Lampiran H).
PENANDA_AM = set("inda madung kehe akkon angkon anso harana bope giot doma milas mur pade halai adaboru oppot "
                 "ngen juo tai sannari soni sattak napedo abope unjung deges sarupo copat satiop manyogot "
                 "natuari tarimokasih".split())
PENANDA_TOBA = set("dang ndang nunga nungga sian naeng jabu alai unang ahu ibana nasida".split())


def token_kata(teks):
    return re.findall(r"\w+", str(teks).lower())


def label_varietas(teks):
    kata = set(token_kata(teks))
    am, tb = bool(kata & PENANDA_AM), bool(kata & PENANDA_TOBA)
    if am:
        return "AM"           # bertanda AM, dengan atau tanpa penanda Toba
    if tb:
        return "Toba_saja"
    return "tak_bertanda"


ARAH = {"id2bbc": ("ind", "bbc"), "bbc2id": ("bbc", "ind")}
print("Utilitas siap.")

---
# Bagian B. Data

## S01. Muat korpus kita dan label arah penerjemahan

**Tujuan:** menyiapkan korpus paralel tingkat kalimat (1.542 pasangan) lengkap dengan label **arah penerjemahan asli** setiap dokumen. Label ini menjadi dasar analisis RQ2.

**Yang dilakukan sel ini:**

1. Mengambil `btb_parallel_sentence.csv` dari folder `input/`. Kalau belum ada, berkas diunduh dari GitHub lalu disimpan ke `input/`, sehingga eksekusi berikutnya memakai berkas yang sama persis.
2. Mengambil label arah. Prioritas pertama adalah `arah_penerjemahan_periksa.xlsx` yang sudah Anda periksa (kolom `arah_final`). Kalau berkas itu belum ada, atau ada baris `arah_final` yang masih kosong, notebook memakai label otomatis (`arah_otomatis`) dan memberi peringatan.
3. Menggabungkan label ke setiap pasangan kalimat, lalu memeriksa teks kosong dan `pair_id` ganda.

**Keluaran yang diharapkan:** `keluaran/s01_korpus.csv` dan tabel jumlah pasangan per arah asli dan genre. Dengan label otomatis, hasilnya kira-kira 1.262 pasangan asli Indonesia dan 280 pasangan asli Toba.

**Istilah:** `asli_toba` berarti teks aslinya ditulis dalam Batak Toba dan sisi Indonesianya terjemahan. `asli_indonesia` berarti sebaliknya.

In [ ]:
# S01 Muat korpus dan label arah
import urllib.request


def unduh_bila_belum(url, tujuan):
    # Unduh sekali, lalu simpan di Drive agar eksekusi berikutnya memakai berkas yang sama.
    tujuan = Path(tujuan)
    if not tujuan.exists():
        print("Mengunduh", url)
        req = urllib.request.Request(url, headers={"User-Agent": "python-urllib"})
        for percobaan in range(4):
            try:
                with urllib.request.urlopen(req, timeout=60) as r:
                    data = r.read()
                break
            except Exception as e:
                if percobaan == 3:
                    raise
                print("  gagal, mencoba lagi:", e)
                time.sleep(2 ** (percobaan + 1))
        sementara = tujuan.with_suffix(tujuan.suffix + ".tmp")
        sementara.write_bytes(data)
        os.replace(sementara, tujuan)
    return tujuan


P_KORPUS_MENTAH = unduh_bila_belum(KONFIGURASI["url_korpus"], DIR_INPUT / "btb_parallel_sentence.csv")
P_LABEL_TERPERIKSA = DIR_INPUT / KONFIGURASI["berkas_label_arah_terperiksa"]
P_LABEL_OTOMATIS = unduh_bila_belum(KONFIGURASI["url_label_arah_otomatis"], DIR_INPUT / "arah_penerjemahan_otomatis.csv")
masukan_s01 = [P_KORPUS_MENTAH, P_LABEL_OTOMATIS] + ([P_LABEL_TERPERIKSA] if P_LABEL_TERPERIKSA.exists() else [])

P_S01 = DIR_KELUARAN / "s01_korpus.csv"
tahap = Tahap("S01", "Korpus dan label arah", masukan_s01, [P_S01], {})
if not tahap.bisa_dilewati():
    korpus = baca_csv(P_KORPUS_MENTAH)

    # --- label arah: otomatis sebagai dasar, ditimpa label terperiksa bila ada ---
    label = baca_csv(P_LABEL_OTOMATIS)[["doc_id", "arah_otomatis"]]
    label["arah_asli"] = label["arah_otomatis"]
    label["sumber_label_arah"] = "otomatis"
    if P_LABEL_TERPERIKSA.exists():
        periksa = pd.read_excel(P_LABEL_TERPERIKSA, sheet_name="Label", dtype=str).fillna("")
        periksa = periksa[["doc_id", "arah_final"]]
        label = label.merge(periksa, on="doc_id", how="left")
        terisi = label["arah_final"].fillna("").str.strip() != ""
        label.loc[terisi, "arah_asli"] = label.loc[terisi, "arah_final"].str.strip()
        label.loc[terisi, "sumber_label_arah"] = "terperiksa"
        print(f"Label terperiksa dipakai untuk {terisi.sum()} dari {len(label)} dokumen.")
    else:
        print("PERINGATAN: arah_penerjemahan_periksa.xlsx belum ada di folder input/. "
              "Label arah memakai tebakan otomatis. Hasil RQ2 bersifat sementara.")
    nilai_sah = {"asli_toba", "asli_indonesia", "campuran", "tidak_diketahui"}
    salah = sorted(set(label["arah_asli"]) - nilai_sah)
    assert not salah, f"Nilai arah tidak dikenal: {salah}. Pakai salah satu dari {sorted(nilai_sah)}."

    korpus = korpus.merge(label[["doc_id", "arah_asli", "sumber_label_arah"]], on="doc_id", how="left")
    korpus = korpus.rename(columns={"toba_final": "teks_bbc", "indo_final": "teks_id"})
    korpus = korpus[["pair_id", "doc_id", "genre", "sub_genre", "teks_id", "teks_bbc", "arah_asli",
                     "sumber_label_arah", "review_status", "validasi_manusia"]]

    # --- pemeriksaan ---
    assert korpus["pair_id"].is_unique, "pair_id ganda"
    assert korpus["arah_asli"].notna().all(), "Ada dokumen tanpa label arah"
    kosong = (korpus["teks_id"].fillna("").str.strip() == "") | (korpus["teks_bbc"].fillna("").str.strip() == "")
    assert not kosong.any(), f"{kosong.sum()} pasangan berteks kosong"

    simpan_csv(korpus, P_S01)
    tabel = pd.crosstab(korpus["genre"], korpus["arah_asli"], margins=True, margins_name="Total")
    tampilkan(tabel, 20)
    tahap.selesai({"pasangan": len(korpus), "dokumen": korpus["doc_id"].nunique(),
                   "per_arah": korpus["arah_asli"].value_counts().to_dict(),
                   "label_terperiksa": int((korpus["sumber_label_arah"] == "terperiksa").sum())})

## S02. Unduh NusaX dan NusaTranslation

**Tujuan:** menyiapkan dua korpus pembanding yang dipakai sebagai data latih dan data uji.

**Yang dilakukan sel ini:**

1. Mengunduh NusaX (kolom `toba_batak`, label `bbc`) dan NusaTranslation (berkas `btk`) dari GitHub IndoNLP, lalu menyimpan berkas mentahnya di `input/`.
2. Mengubah keduanya ke format yang sama: `uji_id`, `split`, `teks_id`, `teks_bbc`.

**Keluaran yang diharapkan:** NusaX 500/100/400 kalimat dan NusaTranslation 6.600/849/2.000 kalimat (latih/validasi/uji), tersimpan di `keluaran/s02_nusax.csv` dan `keluaran/s02_nusatranslation.csv`.

**Catatan:** di notebook ini sisi Batak kedua korpus tetap ditulis `teks_bbc` agar seragam. Hasil S03 menunjukkan bahwa sisi Batak NusaTranslation tidak seluruhnya Batak Toba.

In [ ]:
# S02 Unduh NusaX dan NusaTranslation
SPLIT_RESMI = ["train", "valid", "test"]
P_NX_MENTAH = {s: unduh_bila_belum(KONFIGURASI["url_nusax"].format(split=s), DIR_INPUT / f"nusax_mt_{s}.csv")
               for s in SPLIT_RESMI}
P_NT_MENTAH = {s: unduh_bila_belum(KONFIGURASI["url_nusatranslation"].format(split=s), DIR_INPUT / f"nusatranslation_btk_{s}.csv")
               for s in SPLIT_RESMI}
P_S02_NX = DIR_KELUARAN / "s02_nusax.csv"
P_S02_NT = DIR_KELUARAN / "s02_nusatranslation.csv"

tahap = Tahap("S02", "NusaX dan NusaTranslation", list(P_NX_MENTAH.values()) + list(P_NT_MENTAH.values()),
              [P_S02_NX, P_S02_NT], {"kolom_nusax": KONFIGURASI["kolom_nusax_batak"]})
if not tahap.bisa_dilewati():
    nama_split = {"train": "latih", "valid": "validasi", "test": "uji"}
    nx, nt = [], []
    for s in SPLIT_RESMI:
        d = pd.read_csv(P_NX_MENTAH[s], dtype=str, keep_default_na=False)
        d = d[d[KONFIGURASI["kolom_nusax_batak"]].str.strip() != ""].reset_index(drop=True)
        nx.append(pd.DataFrame({"uji_id": [f"nx-{s}-{i:04d}" for i in range(len(d))], "split": nama_split[s],
                                "teks_id": d["indonesian"], "teks_bbc": d[KONFIGURASI["kolom_nusax_batak"]]}))
        d = pd.read_csv(P_NT_MENTAH[s], dtype=str, keep_default_na=False)
        nt.append(pd.DataFrame({"uji_id": [f"nt-{s}-{i:04d}" for i in range(len(d))], "id_asli": d["id"],
                                "split": nama_split[s], "teks_id": d["ind_text"], "teks_bbc": d["tgt_text"]}))
    nx, nt = pd.concat(nx, ignore_index=True), pd.concat(nt, ignore_index=True)
    simpan_csv(nx, P_S02_NX)
    simpan_csv(nt, P_S02_NT)
    tahap.selesai({"nusax": nx["split"].value_counts().to_dict(),
                   "nusatranslation": nt["split"].value_counts().to_dict()})

## S03. Audit varietas dengan penanda leksikal (RQ1)

**Tujuan:** memeriksa apakah sisi Batak setiap korpus memang Batak Toba, memakai protokol audit dari naskah kami sebelumnya.

**Cara kerja:** setiap kalimat Batak diberi salah satu dari tiga label:

- `AM`: memuat setidaknya satu dari 32 penanda Angkola-Mandailing, misalnya negasi *inda*, *harana* 'karena', *anso* 'supaya'.
- `Toba_saja`: tidak memuat penanda AM, tetapi memuat setidaknya satu dari 12 penanda Toba, misalnya negasi *dang* dan *ndang*.
- `tak_bertanda`: tidak memuat penanda keduanya.

Penanda ini bersifat **heuristik**. Pada naskah sebelumnya, presisi label AM yang diverifikasi penutur asli adalah 0,98, dan 14% kalimat tak bertanda ternyata juga bukan Toba. Jadi, angka AM di sini adalah **batas bawah**.

**Keluaran yang diharapkan:**

- `s03_audit_varietas.csv`: persentase setiap label per korpus dan split. Perkiraan: korpus kita sekitar 0,3% AM, NusaX sekitar 0%, NusaTranslation sekitar 23%.
- `s03_audit_korpus_kami.csv`: label per genre dan arah asli di korpus kita.
- `s03_contoh_am_korpus_kami.csv`: kalimat bertanda AM di korpus kita untuk diperiksa manual. Bisa jadi itu salah tanda, misalnya *tai* atau *soni* yang juga dipakai di Toba.
- Gambar `gambar/s03_audit_varietas.png`.

In [ ]:
# S03 Audit varietas
import matplotlib
import matplotlib.pyplot as plt

P_S01 = DIR_KELUARAN / "s01_korpus.csv"
P_S03 = DIR_KELUARAN / "s03_audit_varietas.csv"
P_S03_KAMI = DIR_KELUARAN / "s03_audit_korpus_kami.csv"
P_S03_CONTOH = DIR_KELUARAN / "s03_contoh_am_korpus_kami.csv"
P_S03_GAMBAR = DIR_GAMBAR / "s03_audit_varietas.png"

tahap = Tahap("S03", "Audit varietas", [P_S01, P_S02_NX, P_S02_NT], [P_S03, P_S03_KAMI, P_S03_CONTOH, P_S03_GAMBAR],
              {"am": sorted(PENANDA_AM), "toba": sorted(PENANDA_TOBA)})
if not tahap.bisa_dilewati():
    korpus = baca_csv(P_S01)
    nx, nt = baca_csv(P_S02_NX), baca_csv(P_S02_NT)
    korpus["varietas"] = korpus["teks_bbc"].map(label_varietas)
    nx["varietas"] = nx["teks_bbc"].map(label_varietas)
    nt["varietas"] = nt["teks_bbc"].map(label_varietas)
    urutan = ["AM", "Toba_saja", "tak_bertanda"]

    def persentase(df, nama, split):
        p = df["varietas"].value_counts(normalize=True).reindex(urutan, fill_value=0) * 100
        return {"korpus": nama, "split": split, "n": len(df), **{f"persen_{k}": round(v, 2) for k, v in p.items()}}

    baris = [persentase(korpus, "Korpus kami", "semua")]
    for nama, df in [("NusaX (bbc)", nx), ("NusaTranslation (btk)", nt)]:
        for s in ["latih", "validasi", "uji"]:
            baris.append(persentase(df[df["split"] == s], nama, s))
        baris.append(persentase(df, nama, "semua"))
    audit = pd.DataFrame(baris)
    simpan_csv(audit, P_S03)
    tampilkan(audit, 20)

    # Rincian korpus kita per genre dan arah asli
    rinci = []
    for kolom in ["genre", "arah_asli"]:
        for nilai, g in korpus.groupby(kolom):
            rinci.append({"dimensi": kolom, **persentase(g, "Korpus kami", nilai)})
    rinci = pd.DataFrame(rinci).rename(columns={"split": "nilai"})
    simpan_csv(rinci, P_S03_KAMI)
    tampilkan(rinci, 20)

    # Kalimat bertanda AM di korpus kita, untuk diperiksa manual
    contoh = korpus[korpus["varietas"] == "AM"].copy()
    contoh["penanda_ditemukan"] = contoh["teks_bbc"].map(lambda t: ", ".join(sorted(set(token_kata(t)) & PENANDA_AM)))
    simpan_csv(contoh[["pair_id", "doc_id", "genre", "penanda_ditemukan", "teks_bbc", "teks_id"]], P_S03_CONTOH)
    print(f"Kalimat bertanda AM di korpus kami: {len(contoh)}")
    tampilkan(contoh[["pair_id", "penanda_ditemukan", "teks_bbc"]], 20)

    # Gambar: persentase AM dan Toba_saja per korpus (semua split)
    g = audit[audit["split"] == "semua"].set_index("korpus")[["persen_AM", "persen_Toba_saja"]]
    ax = g.plot(kind="bar", figsize=(7, 4), color=["#D98A1C", "#3B6FD8"], rot=0)
    ax.set_ylabel("% kalimat")
    ax.set_xlabel("")
    ax.set_title("Kalimat bertanda Angkola-Mandailing (AM) dan Toba")
    ax.legend(["Bertanda AM", "Hanya bertanda Toba"])
    plt.tight_layout()
    plt.savefig(P_S03_GAMBAR, dpi=200)
    plt.show()

    nilai = audit.set_index(["korpus", "split"])["persen_AM"]
    tahap.selesai({"AM_korpus_kami_persen": nilai[("Korpus kami", "semua")],
                   "AM_nusax_persen": nilai[("NusaX (bbc)", "semua")],
                   "AM_nusatranslation_uji_persen": nilai[("NusaTranslation (btk)", "uji")],
                   "kalimat_AM_korpus_kami": len(contoh)})

## S04. Pembagian data berbasis dokumen dan cek tumpang tindih

**Tujuan:** membagi korpus kita menjadi data latih, validasi, dan uji **berdasarkan dokumen**. Semua kalimat dari satu dokumen masuk ke split yang sama, sehingga model tidak "mengintip" konteks dokumen uji saat pelatihan.

**Cara kerja:**

1. Dokumen dikelompokkan per **strata** (lapisan): arah asli × genre.
2. Di setiap strata, urutan dokumen diacak dengan benih tetap. Dokumen lalu dimasukkan ke set uji sampai kira-kira 20% pasangan strata itu, kemudian ke set validasi sampai 10%. Sisanya menjadi data latih. Dokumen besar yang membuat set uji jauh melebihi target (lebih dari `toleransi_lebih` × target) dilewati ke split berikutnya.
3. **Cek tumpang tindih:** kalimat yang sama persis, atau hampir sama (*Jaccard index* atau indeks Jaccard token ≥ 0,9) pada salah satu sisi, dicari antara data latih satu korpus dan data uji korpus lain. Kalimat latih yang tumpang tindih dengan data uji dibuang dari data latih.

**Keluaran yang diharapkan:**

- `s04_split_korpus_kami.csv`: `pair_id` → `split`.
- `s04_ringkasan_split.csv`: jumlah pasangan dan dokumen per split, arah asli, dan genre. Targetnya sekitar 1.080 latih, 150 validasi, dan 310 uji. Angka pastinya bergantung pada ukuran dokumen.
- `s04_tumpang_tindih.csv`: daftar pasangan tumpang tindih yang dibuang dari data latih. Kemungkinan besar kosong atau sangat sedikit.

In [ ]:
# S04 Pembagian data dan cek tumpang tindih
P_S04_SPLIT = DIR_KELUARAN / "s04_split_korpus_kami.csv"
P_S04_RINGKAS = DIR_KELUARAN / "s04_ringkasan_split.csv"
P_S04_TUMPANG = DIR_KELUARAN / "s04_tumpang_tindih.csv"
param_s04 = {**KONFIGURASI["pembagian"], "benih": KONFIGURASI["benih_global"],
             "ambang_jaccard": KONFIGURASI["ambang_jaccard_duplikat"]}


def normalisasi(teks):
    return " ".join(token_kata(teks))


def cari_tumpang_tindih(df_a, df_b, ambang):
    # Pasangan (a, b) yang sama persis atau Jaccard token >= ambang pada salah satu sisi.
    # Memakai prefix filtering agar cepat: kandidat harus memuat salah satu token terlangka dari kalimat a.
    hasil = []
    for sisi in ["teks_id", "teks_bbc"]:
        set_a = [set(token_kata(t)) for t in df_a[sisi]]
        set_b = [set(token_kata(t)) for t in df_b[sisi]]
        frek = {}
        for s in set_a + set_b:
            for w in s:
                frek[w] = frek.get(w, 0) + 1
        indeks = {}
        for j, s in enumerate(set_b):
            for w in s:
                indeks.setdefault(w, []).append(j)
        for i, sa in enumerate(set_a):
            if not sa:
                continue
            urut = sorted(sa, key=lambda w: (frek[w], w))
            p = len(sa) - math.ceil(ambang * len(sa)) + 1
            kandidat = set()
            for w in urut[:p]:
                kandidat.update(indeks.get(w, []))
            for j in kandidat:
                sb = set_b[j]
                jac = len(sa & sb) / len(sa | sb)
                if jac >= ambang:
                    hasil.append({"id_a": df_a["uji_id"].iat[i], "id_b": df_b["uji_id"].iat[j],
                                  "sisi": sisi, "jaccard": round(jac, 3)})
    return pd.DataFrame(hasil, columns=["id_a", "id_b", "sisi", "jaccard"])


tahap = Tahap("S04", "Pembagian data", [P_S01, P_S02_NX, P_S02_NT], [P_S04_SPLIT, P_S04_RINGKAS, P_S04_TUMPANG], param_s04)
if not tahap.bisa_dilewati():
    korpus = baca_csv(P_S01)
    rng = random.Random(param_s04["benih"])
    dok = korpus.groupby("doc_id").agg(n=("pair_id", "size"), genre=("genre", "first"),
                                       arah=("arah_asli", "first")).reset_index()
    dok["strata"] = dok["arah"] + " | " + dok["genre"]
    split_dok = {}
    for strata, g in dok.groupby("strata"):
        ids = sorted(g["doc_id"])
        rng.shuffle(ids)
        total = g["n"].sum()
        target = {"uji": param_s04["porsi_uji"] * total, "validasi": param_s04["porsi_validasi"] * total}
        isi = {"uji": 0, "validasi": 0}
        for d in ids:
            n = int(g.loc[g["doc_id"] == d, "n"].iat[0])
            tujuan = "latih"
            for s in ["uji", "validasi"]:
                if isi[s] < target[s] and isi[s] + n <= max(target[s] * param_s04["toleransi_lebih"], 1):
                    tujuan = s
                    break
            split_dok[d] = tujuan
            if tujuan != "latih":
                isi[tujuan] += n
    korpus["split"] = korpus["doc_id"].map(split_dok)
    korpus["uji_id"] = "kami-" + korpus["pair_id"]

    # --- cek tumpang tindih lintas korpus ---
    nx, nt = baca_csv(P_S02_NX), baca_csv(P_S02_NT)
    ambang = param_s04["ambang_jaccard"]
    periksa = [
        ("kami_latih vs nusax_uji", korpus[korpus.split == "latih"], nx[nx.split == "uji"]),
        ("kami_latih vs nusatranslation_uji", korpus[korpus.split == "latih"], nt[nt.split == "uji"]),
        ("nusax_latih vs kami_uji", nx[nx.split == "latih"], korpus[korpus.split == "uji"]),
        ("nusatranslation_latih vs kami_uji", nt[nt.split == "latih"], korpus[korpus.split == "uji"]),
    ]
    tumpang = []
    for nama, a, b in periksa:
        t = cari_tumpang_tindih(a.reset_index(drop=True), b.reset_index(drop=True), ambang)
        t.insert(0, "pemeriksaan", nama)
        tumpang.append(t)
        print(f"{nama}: {t['id_a'].nunique()} kalimat latih tumpang tindih")
    tumpang = pd.concat(tumpang, ignore_index=True)
    simpan_csv(tumpang, P_S04_TUMPANG)

    simpan_csv(korpus[["pair_id", "uji_id", "doc_id", "split"]], P_S04_SPLIT)
    ringkas = (korpus.groupby(["split", "arah_asli", "genre"])
               .agg(pasangan=("pair_id", "size"), dokumen=("doc_id", "nunique")).reset_index())
    simpan_csv(ringkas, P_S04_RINGKAS)
    print(pd.crosstab(korpus["split"], korpus["arah_asli"], margins=True, margins_name="Total"))
    tahap.selesai({"pasangan_per_split": korpus["split"].value_counts().to_dict(),
                   "dokumen_per_split": korpus.groupby("split")["doc_id"].nunique().to_dict(),
                   "kalimat_latih_tumpang_tindih": int(tumpang["id_a"].nunique())})

## S05. Susun set uji dan konfigurasi data latih

**Tujuan:** menyiapkan satu tabel set uji dan satu berkas per konfigurasi data latih, sehingga semua sistem di bagian C memakai data yang sama persis.

**Set uji** (masing-masing diterjemahkan ke dua arah, `id2bbc` dan `bbc2id`):

| Nama | Isi | Dipakai oleh |
|---|---|---|
| `kami` | set uji korpus kita (sekitar 310 pasangan) | semua sistem |
| `nusax` | uji NusaX (400) | semua sistem |
| `nusatranslation` | uji NusaTranslation (2.000) | NLLB dan MADLAD |
| `nusatranslation_toba` | subsampel 400 kalimat uji NusaTranslation berlabel `Toba_saja` | LLM; juga dihitung untuk sistem lain agar bisa dibandingkan |

**Konfigurasi data latih** (S08): `nusax`, `nusatranslation`, `kami`, dan gabungannya. Kalimat latih yang tumpang tindih dengan data uji (S04) sudah dibuang.

**Keluaran yang diharapkan:** `s05_uji.csv`, `s05_validasi.csv`, dan `s05_latih_<konfigurasi>.csv`, serta tabel jumlah kalimat per set.

In [ ]:
# S05 Susun set uji dan data latih
P_S05_UJI = DIR_KELUARAN / "s05_uji.csv"
P_S05_VALID = DIR_KELUARAN / "s05_validasi.csv"
NAMA_DATA_LATIH = ["nusax", "nusatranslation", "kami"]
P_S05_LATIH = {n: DIR_KELUARAN / f"s05_latih_{n}.csv" for n in NAMA_DATA_LATIH}
param_s05 = {"sampel_nt_toba": KONFIGURASI["llm"]["sampel_nusatranslation_toba"], "benih": KONFIGURASI["benih_global"]}

tahap = Tahap("S05", "Set uji dan data latih", [P_S01, P_S02_NX, P_S02_NT, P_S04_SPLIT, P_S04_TUMPANG],
              [P_S05_UJI, P_S05_VALID] + list(P_S05_LATIH.values()), param_s05)
if not tahap.bisa_dilewati():
    korpus = baca_csv(P_S01).merge(baca_csv(P_S04_SPLIT)[["pair_id", "uji_id", "split"]], on="pair_id")
    nx, nt = baca_csv(P_S02_NX), baca_csv(P_S02_NT)
    buang = set(baca_csv(P_S04_TUMPANG)["id_a"])

    kolom = ["uji_id", "set_uji", "teks_id", "teks_bbc", "arah_asli", "genre", "varietas_rujukan", "validasi_manusia"]
    uji_kami = korpus[korpus.split == "uji"].assign(set_uji="kami")
    uji_nx = nx[nx.split == "uji"].assign(set_uji="nusax", arah_asli="asli_indonesia", genre="ulasan dan komentar",
                                          validasi_manusia="")
    uji_nt = nt[nt.split == "uji"].assign(set_uji="nusatranslation", arah_asli="asli_indonesia",
                                          genre="ulasan dan cuitan", validasi_manusia="")
    uji = pd.concat([uji_kami, uji_nx, uji_nt], ignore_index=True)
    uji["varietas_rujukan"] = uji["teks_bbc"].map(label_varietas)
    uji = uji[kolom]

    # Subsampel uji NusaTranslation yang hanya bertanda Toba (untuk LLM)
    kandidat = uji[(uji.set_uji == "nusatranslation") & (uji.varietas_rujukan == "Toba_saja")]
    n_sampel = min(param_s05["sampel_nt_toba"], len(kandidat))
    sampel = kandidat.sample(n=n_sampel, random_state=param_s05["benih"]).assign(set_uji="nusatranslation_toba")
    uji = pd.concat([uji, sampel], ignore_index=True)
    simpan_csv(uji, P_S05_UJI)

    validasi = pd.concat([korpus[korpus.split == "validasi"].assign(sumber="kami"),
                          nx[nx.split == "validasi"].assign(sumber="nusax")], ignore_index=True)
    simpan_csv(validasi[["uji_id", "sumber", "teks_id", "teks_bbc"]], P_S05_VALID)

    latih = {"kami": korpus[korpus.split == "latih"], "nusax": nx[nx.split == "latih"],
             "nusatranslation": nt[nt.split == "latih"]}
    jumlah = {}
    for nama, df in latih.items():
        df = df[~df["uji_id"].isin(buang)].assign(sumber=nama)
        simpan_csv(df[["uji_id", "sumber", "teks_id", "teks_bbc"]], P_S05_LATIH[nama])
        jumlah[nama] = len(df)

    print(uji.groupby("set_uji").size().rename("kalimat").to_string())
    print("Data latih:", jumlah)
    tahap.selesai({"uji": uji.groupby("set_uji").size().to_dict(), "latih": jumlah, "validasi": len(validasi)})


def muat_set_uji(nama):
    uji = baca_csv(P_S05_UJI)
    return uji[uji["set_uji"] == nama].reset_index(drop=True)


def muat_data_latih(konfigurasi):
    # "kami+nusax" -> gabungan berkas latih kami dan nusax
    return pd.concat([baca_csv(P_S05_LATIH[n]) for n in konfigurasi.split("+")], ignore_index=True)

---
# Bagian C. Eksperimen model

## C.0 Fungsi bantu model

**Yang dilakukan sel ini:** mendefinisikan fungsi yang dipakai S06 sampai S09. Sel ini **tidak memuat model apa pun**, jadi cepat dijalankan.

- `muat_nllb`, `muat_madlad`, `muat_llm`: memuat model dan *tokenizer* dari Hugging Face dengan presisi yang sesuai GPU. A100 memakai `bfloat16`. T4 memakai `float16` untuk NLLB dan `float32` untuk MADLAD (model T5 sering menghasilkan NaN pada `float16`). LLM di GPU kecil dimuat dalam 4-bit.
- `jalankan_unit`: menerjemahkan satu set uji pada satu arah, potong demi potong, dan menulis hasilnya langsung ke Drive. Kalau terputus, eksekusi berikutnya melanjutkan dari kalimat yang belum diterjemahkan.
- `terjemah_nllb`: *beam search* (pencarian berkas) dengan *beam* 5. Kode bahasa sasaran dipaksa sebagai token pertama (`forced_bos_token_id`). Kalau memori GPU habis, potongan dibelah dua secara otomatis.

**Keluaran yang diharapkan:** `Fungsi model siap. Perangkat: cuda` (atau `cpu` bila tanpa GPU).

In [ ]:
# C.0 Fungsi bantu model
from tqdm.auto import tqdm

try:
    import torch
    PERANGKAT = "cuda" if torch.cuda.is_available() else "cpu"
    BF16 = PERANGKAT == "cuda" and torch.cuda.is_bf16_supported()
    MEMORI_GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if PERANGKAT == "cuda" else 0
except ImportError:
    torch, PERANGKAT, BF16, MEMORI_GPU_GB = None, "cpu", False, 0


def ambil_token_hf():
    # Token dibaca dari menu Secrets Colab (nama: HF_TOKEN). Jangan menulis token di sel.
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ.get("HF_TOKEN")


HF_TOKEN = ambil_token_hf()


def argumen_dtype(dtype):
    # transformers >= 4.56 memakai argumen "dtype"; versi lama memakai "torch_dtype".
    import transformers
    versi = tuple(int(x) for x in re.findall(r"\d+", transformers.__version__)[:2])
    return {"dtype": dtype} if versi >= (4, 56) else {"torch_dtype": dtype}


def nama_pendek(nama_model):
    # "facebook/nllb-200-distilled-600M" -> "nllb600m"; "google/madlad400-3b-mt" -> "madlad3b"
    dasar = nama_model.split("/")[-1].lower()
    if "nllb" in dasar:
        return "nllb" + re.findall(r"(\d+(?:\.\d+)?[mb])$", dasar)[0]
    if "madlad" in dasar:
        return "madlad" + re.findall(r"-(\d+b)", dasar)[0]
    return re.sub(r"[^a-z0-9.\-]+", "-", dasar)


def muat_nllb(nama_model):
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    tok = AutoTokenizer.from_pretrained(nama_model, token=HF_TOKEN)
    dtype = torch.bfloat16 if BF16 else (torch.float16 if PERANGKAT == "cuda" else torch.float32)
    model = AutoModelForSeq2SeqLM.from_pretrained(nama_model, token=HF_TOKEN, **argumen_dtype(dtype)).to(PERANGKAT)
    model.eval()
    return tok, model


def muat_madlad(nama_model):
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    tok = AutoTokenizer.from_pretrained(nama_model, token=HF_TOKEN)
    dtype = torch.bfloat16 if BF16 else torch.float32   # T5 tidak stabil di float16
    model = AutoModelForSeq2SeqLM.from_pretrained(nama_model, token=HF_TOKEN, **argumen_dtype(dtype)).to(PERANGKAT)
    model.eval()
    return tok, model


def muat_llm(nama_model):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(nama_model, token=HF_TOKEN)
    tok.padding_side = "left"            # LLM menghasilkan teks di sebelah kanan, jadi padding di kiri
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    argumen = {"token": HF_TOKEN, "device_map": "auto"}
    if MEMORI_GPU_GB and MEMORI_GPU_GB < 30:
        # GPU kecil (misalnya T4 15 GB): muat dalam 4-bit agar model 7B sampai 9B muat.
        pasang_bila_perlu("bitsandbytes", "bitsandbytes")
        from transformers import BitsAndBytesConfig
        argumen["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                            bnb_4bit_compute_dtype=torch.float16)
    else:
        argumen.update(argumen_dtype(torch.bfloat16 if BF16 else torch.float32))
    model = AutoModelForCausalLM.from_pretrained(nama_model, **argumen)
    model.eval()
    return tok, model


def sumber_dan_rujukan(df, arah):
    if arah == "id2bbc":
        return df["teks_id"], df["teks_bbc"]
    return df["teks_bbc"], df["teks_id"]


def jalankan_unit(sistem, set_uji, arah, parameter, fungsi_terjemah, ukuran_potong):
    # Menerjemahkan kalimat yang belum ada di berkas prediksi, potong demi potong.
    df = muat_set_uji(set_uji)
    unit = UnitPrediksi(sistem, set_uji, arah, parameter)
    sisa = unit.sisa(df)
    if len(sisa) == 0:
        print(f"  {sistem} | {set_uji} | {arah}: sudah lengkap ({len(df)} kalimat)")
        return
    sumber, _ = sumber_dan_rujukan(sisa, arah)
    urutan = np.argsort([-len(s) for s in sumber.values], kind="stable")  # terpanjang dulu: OOM ketahuan di awal
    ids, teks = sisa["uji_id"].values[urutan], sumber.values[urutan]
    print(f"  {sistem} | {set_uji} | {arah}: {len(ids)} dari {len(df)} kalimat perlu diterjemahkan")
    for mulai in tqdm(range(0, len(ids), ukuran_potong), leave=False):
        potong = list(teks[mulai:mulai + ukuran_potong])
        unit.tambah(list(ids[mulai:mulai + ukuran_potong]), fungsi_terjemah(potong, arah))


def unit_lengkap(sistem, set_uji, arah, parameter):
    return UnitPrediksi(sistem, set_uji, arah, parameter, buat=False).lengkap(muat_set_uji(set_uji))


def dengan_belah_dua(fungsi):
    # Bila memori GPU habis, potongan dibelah dua lalu dicoba lagi.
    def bungkus(teks, *args, **kwargs):
        try:
            return fungsi(teks, *args, **kwargs)
        except torch.cuda.OutOfMemoryError:
            bersihkan_gpu()
            if len(teks) == 1:
                raise
            tengah = len(teks) // 2
            return bungkus(teks[:tengah], *args, **kwargs) + bungkus(teks[tengah:], *args, **kwargs)
    return bungkus


def kode_nllb(sisi):
    return KONFIGURASI["kode_nllb_indonesia"] if sisi == "ind" else KONFIGURASI["kode_nllb_batak"]


@dengan_belah_dua
def terjemah_nllb(teks, arah, tok, model):
    c = KONFIGURASI["dekode"]
    sisi_sumber, sisi_sasaran = ARAH[arah]
    tok.src_lang = kode_nllb(sisi_sumber)
    enc = tok(teks, return_tensors="pt", padding=True, truncation=True, max_length=c["panjang_maks_sumber"]).to(model.device)
    maks = min(c["panjang_maks_keluaran"], int(enc["input_ids"].shape[1] * c["faktor_panjang_keluaran"] + c["tambahan_panjang"]))
    with torch.inference_mode():
        keluar = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(kode_nllb(sisi_sasaran)),
                                num_beams=c["beam"], max_new_tokens=maks)
    return [t.strip() for t in tok.batch_decode(keluar, skip_special_tokens=True)]


def periksa_kode_bahasa(tok, kode):
    # Kode yang tidak ada di tokenizer diubah diam-diam menjadi <unk>. Ini harus dicegah.
    return tok.convert_tokens_to_ids(kode) != tok.unk_token_id


def param_dekode_nllb(nama_model):
    return {"model": nama_model, "dekode": KONFIGURASI["dekode"], "kode_id": KONFIGURASI["kode_nllb_indonesia"],
            "kode_bbc": KONFIGURASI["kode_nllb_batak"]}


print("Fungsi model siap. Perangkat:", PERANGKAT, "| bfloat16:", BF16, "| memori GPU (GB):", round(MEMORI_GPU_GB, 1))

## S06. *Baseline copy-source* dan NLLB-200 *zero-shot* (RQ3)

**Tujuan:** mengukur titik awal sebelum pelatihan apa pun.

- ***Copy-source*** (salin sumber): "terjemahan" berupa kalimat sumber yang disalin apa adanya. Karena Batak Toba dan Bahasa Indonesia berbagi banyak kata serapan dan nama diri, skor ini **tidak nol**. Skor ini menjadi batas bawah yang harus dilampaui sistem mana pun.
- **NLLB-200 *zero-shot***: model 600M, 1,3B, dan 3,3B parameter tanpa pelatihan tambahan, dengan kode Banjar (`bjn_Latn`) sebagai pengganti kode Batak Toba yang tidak tersedia.

**Yang dilakukan sel ini:** untuk setiap model, sel memeriksa dulu apakah semua prediksinya sudah lengkap. Model hanya dimuat kalau masih ada yang perlu diterjemahkan, lalu dihapus dari memori setelah selesai.

**Keluaran yang diharapkan:** berkas `prediksi/<sistem>/<set_uji>.<arah>.jsonl`, misalnya `prediksi/nllb600m_zeroshot/kami.id2bbc.jsonl`. Pesan `sudah lengkap` berarti unit itu diambil dari *checkpoint*. Skor dihitung nanti di S10.

**Perkiraan:** skor *zero-shot* untuk arah id→bbc biasanya rendah, karena model menulis Banjar atau Indonesia. Arah bbc→id lebih baik, karena model cukup mengenal Bahasa Indonesia.

In [ ]:
# S06 Copy-source dan NLLB zero-shot
P_S06 = DIR_KELUARAN / "s06_ringkasan.json"
tahap = Tahap("S06", "Copy-source dan NLLB zero-shot", [P_S05_UJI], [P_S06],
              {"model": KONFIGURASI["nllb_zero_shot"], "set_uji": KONFIGURASI["set_uji_zero_shot"],
               "dekode": KONFIGURASI["dekode"], "kode_bbc": KONFIGURASI["kode_nllb_batak"]})
if not tahap.bisa_dilewati():
    set_uji_s06 = KONFIGURASI["set_uji_zero_shot"]
    # 1) Copy-source: tidak butuh GPU
    for s in set_uji_s06:
        for arah in ARAH:
            jalankan_unit("copy_source", s, arah, {"sistem": "copy_source"}, lambda teks, arah: list(teks), 1000)

    # 2) NLLB zero-shot
    for nama_model in KONFIGURASI["nllb_zero_shot"]:
        sistem = nama_pendek(nama_model) + "_zeroshot"
        param = param_dekode_nllb(nama_model)
        perlu = [(s, a) for s in set_uji_s06 for a in ARAH if not unit_lengkap(sistem, s, a, param)]
        if not perlu:
            print(f"{sistem}: semua unit sudah lengkap.")
            continue
        print(f"Memuat {nama_model} ...")
        tok, model = muat_nllb(nama_model)
        assert periksa_kode_bahasa(tok, KONFIGURASI["kode_nllb_batak"]), "Kode bahasa Batak pengganti tidak ada di tokenizer"
        for s, a in perlu:
            jalankan_unit(sistem, s, a, param, lambda teks, arah: terjemah_nllb(teks, arah, tok, model),
                          KONFIGURASI["dekode"]["ukuran_batch"])
        del model, tok
        bersihkan_gpu()

    sistem_s06 = ["copy_source"] + [nama_pendek(m) + "_zeroshot" for m in KONFIGURASI["nllb_zero_shot"]]
    tulis_atomik(P_S06, json.dumps({"sistem": sistem_s06, "set_uji": set_uji_s06}, indent=2))
    tahap.selesai({"sistem": sistem_s06})

## S07. MADLAD-400 *zero-shot* (RQ3)

**Tujuan:** menguji model MT yang dilatih pada data web lebih dari 400 bahasa, yang mungkin mencakup Batak Toba.

**Yang dilakukan sel ini:**

1. Memeriksa apakah *tokenizer* MADLAD memiliki token bahasa `<2bbc>`. Kalau tidak ada, model dilewati dan alasannya dicatat. Ini penting supaya kita tidak melaporkan skor dari kode bahasa yang tidak dikenal model.
2. Kalau ada, menerjemahkan semua set uji. Sumber diberi awalan token bahasa sasaran, misalnya `<2bbc> Saya pergi ke pasar.`

**Keluaran yang diharapkan:** `prediksi/madlad3b_zeroshot/...` dan `s07_ringkasan.json` yang mencatat apakah `<2bbc>` tersedia.

**Catatan:** MADLAD-400 3B butuh sekitar 12 GB memori di T4 (`float32`) dan sekitar 6 GB di A100 (`bfloat16`).

In [ ]:
# S07 MADLAD-400 zero-shot
P_S07 = DIR_KELUARAN / "s07_ringkasan.json"


@dengan_belah_dua
def terjemah_madlad(teks, arah, tok, model):
    c = KONFIGURASI["dekode"]
    awalan = KONFIGURASI["kode_madlad_batak"] if arah == "id2bbc" else KONFIGURASI["kode_madlad_indonesia"]
    enc = tok([f"{awalan} {t}" for t in teks], return_tensors="pt", padding=True, truncation=True,
              max_length=c["panjang_maks_sumber"]).to(model.device)
    maks = min(c["panjang_maks_keluaran"], int(enc["input_ids"].shape[1] * c["faktor_panjang_keluaran"] + c["tambahan_panjang"]))
    with torch.inference_mode():
        keluar = model.generate(**enc, num_beams=c["beam"], max_new_tokens=maks)
    return [t.strip() for t in tok.batch_decode(keluar, skip_special_tokens=True)]


tahap = Tahap("S07", "MADLAD-400 zero-shot", [P_S05_UJI], [P_S07],
              {"model": KONFIGURASI["madlad"], "set_uji": KONFIGURASI["set_uji_zero_shot"], "dekode": KONFIGURASI["dekode"],
               "kode": [KONFIGURASI["kode_madlad_indonesia"], KONFIGURASI["kode_madlad_batak"]]})
if not tahap.bisa_dilewati():
    catatan = {}
    for nama_model in KONFIGURASI["madlad"]:
        sistem = nama_pendek(nama_model) + "_zeroshot"
        param = {"model": nama_model, "dekode": KONFIGURASI["dekode"]}
        perlu = [(s, a) for s in KONFIGURASI["set_uji_zero_shot"] for a in ARAH if not unit_lengkap(sistem, s, a, param)]
        if not perlu:
            catatan[sistem] = "lengkap"
            print(f"{sistem}: semua unit sudah lengkap.")
            continue
        print(f"Memuat {nama_model} ...")
        tok, model = muat_madlad(nama_model)
        ada = {k: periksa_kode_bahasa(tok, KONFIGURASI[k]) for k in ["kode_madlad_batak", "kode_madlad_indonesia"]}
        print("Token bahasa tersedia:", ada)
        if not all(ada.values()):
            catatan[sistem] = f"dilewati: token bahasa tidak tersedia {ada}"
            print(f"{sistem} dilewati karena token bahasa tidak tersedia.")
        else:
            for s, a in perlu:
                jalankan_unit(sistem, s, a, param, lambda teks, arah: terjemah_madlad(teks, arah, tok, model),
                              KONFIGURASI["dekode"]["ukuran_batch"])
            catatan[sistem] = "lengkap"
        del model, tok
        bersihkan_gpu()
    tulis_atomik(P_S07, json.dumps(catatan, indent=2, ensure_ascii=False))
    tahap.selesai({"catatan": catatan})

## S08. *Fine-tuning* NLLB-200 pada beberapa konfigurasi data (RQ4)

**Tujuan:** mengukur nilai korpus kita sebagai data latih. Model NLLB-200 600M dilatih ulang pada lima konfigurasi data, lalu diuji pada **ketiga** set uji, termasuk set uji di luar domain data latihnya.

| Konfigurasi | Data latih | Pertanyaan yang dijawab |
|---|---|---|
| `nusax` | NusaX (500) | *baseline* data Toba yang kecil |
| `nusatranslation` | NusaTranslation (6.600) | *baseline* data terbesar (varietas campuran) |
| `kami` | korpus kita (sekitar 1.080) | seberapa jauh korpus kita sendiri membawa model? |
| `kami+nusax` | gabungan | apakah korpus kita menambah nilai pada data Toba lain? |
| `kami+nusatranslation` | gabungan | apakah korpus kita menambah nilai pada data terbesar? |

***Hyperparameter*** mengikuti naskah sebelumnya agar sebanding: Adafactor, laju belajar 1e-4, *batch* efektif 16, 5 *epoch*, jadwal *cosine* dengan *warmup* 10%, panjang maksimum 128 token, *checkpoint* terakhir tanpa seleksi, dan *beam* 5. Satu model dilatih untuk dua arah sekaligus. Kode `bjn_Latn` dipakai untuk Batak Toba. Konfigurasi kecil diulang dengan tiga benih (42, 1, 2) karena variansnya lebih besar.

**Cara kerja *checkpoint* pelatihan:**

1. Setiap `simpan_tiap_langkah` langkah dan di akhir setiap *epoch*, bobot model, keadaan *optimizer*, jadwal laju belajar, dan posisi data disimpan ke `model_sementara/<run>/ckpt.pt`.
2. Kalau sesi terputus, menjalankan sel ini lagi akan melanjutkan dari langkah itu. Urutan data setiap *epoch* ditentukan oleh benih, jadi hasilnya tetap sama.
3. Setelah pelatihan selesai, bobot akhir disimpan (`bobot_akhir.pt`), lalu semua set uji diterjemahkan dengan mekanisme lanjut per kalimat seperti S06.
4. Setelah semua prediksi lengkap, bobot dihapus (bila `hapus_bobot_setelah_selesai=True`) agar Drive tidak penuh. Satu *checkpoint* berukuran sekitar 2,5 GB.

**Keluaran yang diharapkan:**

- Log *loss* (nilai galat) setiap 50 langkah dan *validation loss* (galat validasi) di akhir setiap *epoch*. *Loss* seharusnya turun dari sekitar 6 sampai 8 ke sekitar 1 sampai 2.
- `prediksi/ft_<konfigurasi>_s<benih>/...` untuk setiap set uji dan arah.
- `keluaran/s08_log_pelatihan.csv` berisi riwayat *loss* semua *run*.

In [ ]:
# S08 Fine-tuning NLLB
P_S08 = DIR_KELUARAN / "s08_ringkasan.json"
P_S08_LOG = DIR_KELUARAN / "s08_log_pelatihan.csv"
SET_UJI_FT = ["kami", "nusax", "nusatranslation"]
KOLOM_LOG = ["run", "langkah", "epoch", "loss_latih", "loss_validasi"]


def tokenisasi_arah(tok, df, arah, maks):
    sisi_sumber, sisi_sasaran = ARAH[arah]
    sumber, sasaran = sumber_dan_rujukan(df, arah)
    tok.src_lang, tok.tgt_lang = kode_nllb(sisi_sumber), kode_nllb(sisi_sasaran)
    enc = tok(list(sumber), text_target=list(sasaran), max_length=maks, truncation=True)
    return list(zip(enc["input_ids"], enc["labels"]))


def susun_batch(contoh, id_pad):
    panjang_x = max(len(x) for x, _ in contoh)
    panjang_y = max(len(y) for _, y in contoh)
    x = torch.full((len(contoh), panjang_x), id_pad, dtype=torch.long)
    m = torch.zeros((len(contoh), panjang_x), dtype=torch.long)
    y = torch.full((len(contoh), panjang_y), -100, dtype=torch.long)   # -100 diabaikan oleh loss
    for i, (a, b) in enumerate(contoh):
        x[i, :len(a)] = torch.tensor(a)
        m[i, :len(a)] = 1
        y[i, :len(b)] = torch.tensor(b)
    return {"input_ids": x, "attention_mask": m, "labels": y}


def buat_optimizer(model, laju):
    try:
        from transformers.optimization import Adafactor
        return Adafactor(model.parameters(), lr=laju, scale_parameter=False, relative_step=False, warmup_init=False)
    except ImportError:
        return torch.optim.Adafactor(model.parameters(), lr=laju)


def buat_jadwal(optimizer, langkah_warmup, langkah_total):
    # Warmup linear lalu cosine turun ke nol.
    def faktor(langkah):
        if langkah < langkah_warmup:
            return (langkah + 1) / max(1, langkah_warmup)
        progres = (langkah - langkah_warmup) / max(1, langkah_total - langkah_warmup)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * min(1.0, progres))))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, faktor)


def simpan_checkpoint_latih(path, model, optimizer, jadwal, keadaan):
    # Keadaan generator acak ikut disimpan agar dropout setelah dilanjutkan sama dengan tanpa jeda.
    acak = {"torch": torch.get_rng_state(), "numpy": np.random.get_state(), "python": random.getstate(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}
    sementara = path.with_suffix(".tmp")
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "jadwal": jadwal.state_dict(), "keadaan": keadaan, "acak": acak}, sementara)
    os.replace(sementara, path)


def pulihkan_acak(acak):
    torch.set_rng_state(acak["torch"])
    np.random.set_state(acak["numpy"])
    random.setstate(acak["python"])
    if acak.get("cuda") is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(acak["cuda"])


def hitung_loss_validasi(model, data, id_pad, ukuran):
    model.eval()
    total, n = 0.0, 0
    with torch.inference_mode():
        for i in range(0, len(data), ukuran):
            b = {k: v.to(model.device) for k, v in susun_batch(data[i:i + ukuran], id_pad).items()}
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
                total += model(**b).loss.item() * len(b["input_ids"])
            n += len(b["input_ids"])
    model.train()
    return total / max(1, n)


def latih_satu_run(konfigurasi, benih, c):
    nama_run = f"ft_{konfigurasi}_s{benih}"
    dir_run = DIR_MODEL / nama_run
    dir_run.mkdir(parents=True, exist_ok=True)
    p_ckpt, p_bobot, p_log = dir_run / "ckpt.pt", dir_run / "bobot_akhir.pt", dir_run / "log.csv"
    param_pred = {"run": nama_run, "fine_tuning": {k: v for k, v in c.items() if k not in ("benih", "konfigurasi_data")},
                  "benih": benih, "dekode": KONFIGURASI["dekode"], "kode_bbc": KONFIGURASI["kode_nllb_batak"],
                  "data_latih": {n: sha256_berkas(P_S05_LATIH[n]) for n in konfigurasi.split("+")}}
    perlu = [(s, a) for s in SET_UJI_FT for a in ARAH if not unit_lengkap(nama_run, s, a, param_pred)]
    if not perlu:
        print(f"{nama_run}: semua prediksi sudah lengkap.")
        if c["hapus_bobot_setelah_selesai"]:
            p_bobot.unlink(missing_ok=True)
            p_ckpt.unlink(missing_ok=True)
        return
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    tok = AutoTokenizer.from_pretrained(c["model_dasar"], token=HF_TOKEN)
    assert periksa_kode_bahasa(tok, KONFIGURASI["kode_nllb_batak"])

    if not p_bobot.exists():
        # ---------------- Pelatihan ----------------
        atur_benih(benih)
        model = AutoModelForSeq2SeqLM.from_pretrained(c["model_dasar"], token=HF_TOKEN).to(PERANGKAT)
        model.train()
        latih = muat_data_latih(konfigurasi)
        arah_latih = list(ARAH) if c["dua_arah"] else ["id2bbc"]
        data = [ex for a in arah_latih for ex in tokenisasi_arah(tok, latih, a, c["panjang_maks"])]
        valid_df = baca_csv(P_S05_VALID)
        valid = [ex for a in arah_latih for ex in tokenisasi_arah(tok, valid_df[valid_df.sumber == "kami"], a, c["panjang_maks"])]
        batch_per_epoch = math.ceil(len(data) / c["ukuran_batch"])
        langkah_per_epoch = math.ceil(batch_per_epoch / c["akumulasi_gradien"])
        langkah_total = langkah_per_epoch * c["epoch"]
        optimizer = buat_optimizer(model, c["laju_belajar"])
        jadwal = buat_jadwal(optimizer, int(c["porsi_warmup"] * langkah_total), langkah_total)
        keadaan = {"epoch": 0, "batch": 0, "langkah": 0}
        riwayat = []
        if p_ckpt.exists():
            simpanan = torch.load(p_ckpt, map_location=PERANGKAT, weights_only=False)
            model.load_state_dict(simpanan["model"])
            optimizer.load_state_dict(simpanan["optimizer"])
            jadwal.load_state_dict(simpanan["jadwal"])
            keadaan = simpanan["keadaan"]
            pulihkan_acak(simpanan["acak"])
            riwayat = baca_csv(p_log).to_dict("records") if p_log.exists() and p_log.stat().st_size > 0 else []
            print(f"{nama_run}: melanjutkan dari epoch {keadaan['epoch'] + 1}, langkah {keadaan['langkah']} dari {langkah_total}")
            del simpanan
        else:
            print(f"{nama_run}: mulai pelatihan. Contoh latih: {len(data)}, langkah total: {langkah_total}")
        id_pad = tok.pad_token_id
        jumlah_loss, n_loss = 0.0, 0
        for epoch in range(keadaan["epoch"], c["epoch"]):
            urutan = np.random.RandomState(benih * 1000 + epoch).permutation(len(data))   # urutan tetap per epoch
            mulai_batch = keadaan["batch"] if epoch == keadaan["epoch"] else 0
            progres = tqdm(range(mulai_batch, batch_per_epoch), desc=f"{nama_run} epoch {epoch + 1}", leave=False)
            for bi in progres:
                idx = urutan[bi * c["ukuran_batch"]:(bi + 1) * c["ukuran_batch"]]
                b = {k: v.to(PERANGKAT) for k, v in susun_batch([data[i] for i in idx], id_pad).items()}
                with torch.autocast("cuda", dtype=torch.bfloat16, enabled=BF16):
                    loss = model(**b).loss / c["akumulasi_gradien"]
                loss.backward()
                jumlah_loss += loss.item() * c["akumulasi_gradien"]
                n_loss += 1
                akhir_epoch = bi + 1 == batch_per_epoch
                if (bi + 1) % c["akumulasi_gradien"] == 0 or akhir_epoch:
                    optimizer.step()
                    jadwal.step()
                    optimizer.zero_grad(set_to_none=True)
                    keadaan["langkah"] += 1
                    if keadaan["langkah"] % 50 == 0:
                        progres.set_postfix(loss=round(jumlah_loss / n_loss, 4))
                        riwayat.append({"run": nama_run, "langkah": keadaan["langkah"], "epoch": epoch + 1,
                                        "loss_latih": round(jumlah_loss / n_loss, 5), "loss_validasi": ""})
                        jumlah_loss, n_loss = 0.0, 0
                    if keadaan["langkah"] % c["simpan_tiap_langkah"] == 0 and not akhir_epoch:
                        keadaan.update(epoch=epoch, batch=bi + 1)
                        simpan_checkpoint_latih(p_ckpt, model, optimizer, jadwal, keadaan)
                        simpan_csv(pd.DataFrame(riwayat, columns=KOLOM_LOG), p_log)
            loss_val = hitung_loss_validasi(model, valid, id_pad, c["ukuran_batch"])
            riwayat.append({"run": nama_run, "langkah": keadaan["langkah"], "epoch": epoch + 1,
                            "loss_latih": "", "loss_validasi": round(loss_val, 5)})
            print(f"{nama_run}: epoch {epoch + 1} selesai, loss validasi (korpus kami) = {loss_val:.4f}")
            keadaan.update(epoch=epoch + 1, batch=0)
            simpan_checkpoint_latih(p_ckpt, model, optimizer, jadwal, keadaan)
            simpan_csv(pd.DataFrame(riwayat, columns=KOLOM_LOG), p_log)
        # Simpan bobot akhir dalam presisi setengah agar ringkas, lalu hapus checkpoint pelatihan.
        dtype_simpan = torch.bfloat16 if BF16 else torch.float16
        sementara = p_bobot.with_suffix(".tmp")
        torch.save({k: v.to(dtype_simpan) for k, v in model.state_dict().items()}, sementara)
        os.replace(sementara, p_bobot)
        p_ckpt.unlink(missing_ok=True)
        del optimizer, jadwal
    else:
        print(f"{nama_run}: pelatihan sudah selesai, memuat bobot akhir untuk penerjemahan.")
        model = AutoModelForSeq2SeqLM.from_pretrained(c["model_dasar"], token=HF_TOKEN)
        model.load_state_dict(torch.load(p_bobot, map_location="cpu", weights_only=False))

    # ---------------- Penerjemahan set uji ----------------
    dtype = torch.bfloat16 if BF16 else (torch.float16 if PERANGKAT == "cuda" else torch.float32)
    model = model.to(PERANGKAT, dtype=dtype).eval()
    for s, a in perlu:
        jalankan_unit(nama_run, s, a, param_pred, lambda teks, arah: terjemah_nllb(teks, arah, tok, model),
                      KONFIGURASI["dekode"]["ukuran_batch"])
    del model
    bersihkan_gpu()
    if c["hapus_bobot_setelah_selesai"] and all(unit_lengkap(nama_run, s, a, param_pred) for s in SET_UJI_FT for a in ARAH):
        p_bobot.unlink(missing_ok=True)


c_ft = KONFIGURASI["fine_tuning"]
tahap = Tahap("S08", "Fine-tuning NLLB", [P_S05_UJI, P_S05_VALID] + list(P_S05_LATIH.values()), [P_S08, P_S08_LOG],
              {"fine_tuning": c_ft, "dekode": KONFIGURASI["dekode"], "kode_bbc": KONFIGURASI["kode_nllb_batak"]})
if not tahap.bisa_dilewati():
    run_semua = [(k, b) for k in c_ft["konfigurasi_data"] for b in c_ft["benih"].get(k, [KONFIGURASI["benih_global"]])]
    print(f"{len(run_semua)} run: {[f'ft_{k}_s{b}' for k, b in run_semua]}")
    for k, b in run_semua:
        latih_satu_run(k, b, c_ft)
    log = [baca_csv(DIR_MODEL / f"ft_{k}_s{b}" / "log.csv") for k, b in run_semua
           if (DIR_MODEL / f"ft_{k}_s{b}" / "log.csv").exists()]
    simpan_csv(pd.concat(log, ignore_index=True) if log else pd.DataFrame(columns=KOLOM_LOG), P_S08_LOG)
    tulis_atomik(P_S08, json.dumps({"run": [f"ft_{k}_s{b}" for k, b in run_semua]}, indent=2))
    tahap.selesai({"run": len(run_semua)})

## S09. LLM *zero-shot* dan *few-shot* (RQ3, RQ5)

**Tujuan:** mengukur kemampuan LLM terbuka menerjemahkan Batak Toba, dan menguji apakah contoh dari korpus kita membantu.

**Tiga kondisi:**

| Kondisi | Isi *prompt* |
|---|---|
| `zero_shot` | instruksi dan kalimat sumber saja |
| `few_shot_acak` | ditambah 5 contoh terjemahan yang dipilih acak dari data latih korpus kita |
| `few_shot_retrieval` | ditambah 5 contoh yang **paling mirip** dengan kalimat sumber, diukur dengan *cosine similarity* (kemiripan kosinus) TF-IDF *n-gram* karakter 2 sampai 4 |

Contoh selalu diambil dari **data latih** korpus kita, tidak pernah dari data uji. Untuk set uji NusaX dan NusaTranslation, kondisi *few-shot* menguji apakah korpus kita berguna sebagai bank contoh untuk domain lain.

***Prompt*** ditulis dalam Bahasa Indonesia sebagai satu pesan pengguna. Instruksi sistem tidak dipakai, karena beberapa model (misalnya Gemma) tidak mendukungnya. Decoding memakai *greedy* (tanpa pengambilan acak), dan baris pertama jawaban diambil sebagai terjemahan.

**Model bawaan:** Qwen2.5-7B-Instruct (multibahasa umum) dan Sahabat-AI Gemma2 9B (dilatih lanjut untuk bahasa-bahasa Indonesia). **Periksa kartu model di Hugging Face** untuk memastikan nama, lisensi, dan klaim cakupan bahasanya. Model yang gagal dimuat akan dilewati dengan pesan, tanpa menghentikan notebook.

**Keluaran yang diharapkan:** `prediksi/llm_<model>_<kondisi>/...`, ditambah contoh *prompt* yang dicetak sekali untuk diperiksa.

In [ ]:
# S09 LLM zero-shot dan few-shot
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

P_S09 = DIR_KELUARAN / "s09_ringkasan.json"
NAMA_BAHASA = {"ind": "Bahasa Indonesia", "bbc": "Bahasa Batak Toba"}


def buat_prompt(teks, arah, contoh):
    s, t = ARAH[arah]
    baris = [f"Terjemahkan kalimat {NAMA_BAHASA[s]} berikut ke dalam {NAMA_BAHASA[t]}. "
             f"Tulis hanya terjemahannya dalam satu baris, tanpa penjelasan."]
    if contoh:
        baris.append("\nContoh terjemahan:")
        for cs, ct in contoh:
            baris.append(f"{NAMA_BAHASA[s]}: {cs}\n{NAMA_BAHASA[t]}: {ct}\n")
    baris.append(f"\n{NAMA_BAHASA[s]}: {teks}\n{NAMA_BAHASA[t]}:")
    return "\n".join(baris)


def rapikan_keluaran_llm(teks):
    baris = [b.strip() for b in str(teks).strip().splitlines() if b.strip()]
    if not baris:
        return ""
    t = baris[0]
    t = re.sub(r"^(\*\*)?(bahasa\s+)?(batak\s+toba|indonesia|terjemahan(nya)?)\s*(\*\*)?\s*:\s*", "", t, flags=re.I)
    return t.strip().strip('"').strip("*").strip()


class BankContoh:
    # Bank contoh dari data latih korpus kita, untuk pemilihan acak dan retrieval.
    def __init__(self, df_latih, k, benih):
        self.df, self.k, self.benih = df_latih.reset_index(drop=True), k, benih
        self.vektor = {}
        for arah in ARAH:
            sumber, _ = sumber_dan_rujukan(self.df, arah)
            v = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), lowercase=True)
            self.vektor[arah] = (v, v.fit_transform(sumber))

    def pasangan(self, idx, arah):
        sumber, sasaran = sumber_dan_rujukan(self.df, arah)
        return [(sumber.iat[i], sasaran.iat[i]) for i in idx]

    def acak(self, uji_id, arah):
        rng = random.Random(f"{self.benih}-{uji_id}-{arah}")
        return self.pasangan(rng.sample(range(len(self.df)), self.k), arah)

    def retrieval(self, teks, arah):
        v, m = self.vektor[arah]
        skor = linear_kernel(v.transform([teks]), m).ravel()
        idx = np.argsort(-skor, kind="stable")[:self.k][::-1]   # yang paling mirip diletakkan paling akhir
        return self.pasangan(idx, arah)


def terjemah_llm(prompts, tok, model, token_baru_maks):
    teks_chat = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False, add_generation_prompt=True)
                 for p in prompts]
    enc = tok(teks_chat, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        keluar = model.generate(**enc, max_new_tokens=token_baru_maks, do_sample=False, pad_token_id=tok.pad_token_id)
    jawaban = tok.batch_decode(keluar[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    return [rapikan_keluaran_llm(j) for j in jawaban]


c_llm = KONFIGURASI["llm"]
tahap = Tahap("S09", "LLM zero-shot dan few-shot", [P_S05_UJI, P_S05_LATIH["kami"]], [P_S09], {"llm": c_llm})
if not tahap.bisa_dilewati():
    bank = BankContoh(baca_csv(P_S05_LATIH["kami"]), c_llm["k_contoh"], KONFIGURASI["benih_global"])
    data_uji = {s: muat_set_uji(s) for s in c_llm["set_uji"]}
    contoh_dicetak = False
    catatan = {}
    for nama_model in c_llm["model"]:
        param = {"model": nama_model, "k": c_llm["k_contoh"], "token_baru_maks": c_llm["token_baru_maks"],
                 "data_latih": sha256_berkas(P_S05_LATIH["kami"]), "versi_prompt": 1}
        perlu = [(kd, s, a) for kd in c_llm["kondisi"] for s in c_llm["set_uji"] for a in ARAH
                 if not unit_lengkap(f"llm_{nama_pendek(nama_model)}_{kd}", s, a, {**param, "kondisi": kd})]
        if not perlu:
            catatan[nama_model] = "lengkap"
            print(f"{nama_model}: semua unit sudah lengkap.")
            continue
        try:
            print(f"Memuat {nama_model} ...")
            tok, model = muat_llm(nama_model)
        except Exception as e:
            catatan[nama_model] = f"gagal dimuat: {type(e).__name__}: {e}"
            print("  Dilewati:", catatan[nama_model])
            continue
        for kd, s, a in perlu:
            sistem = f"llm_{nama_pendek(nama_model)}_{kd}"
            df = data_uji[s]

            def fungsi(teks, arah, kd=kd, df=df):
                # Untuk contoh acak, benihnya memakai uji_id agar contoh tiap kalimat tetap sama antar-eksekusi.
                sumber, _ = sumber_dan_rujukan(df, arah)
                peta = dict(zip(sumber, df["uji_id"]))
                prompts = []
                for t in teks:
                    if kd == "zero_shot":
                        contoh = []
                    elif kd == "few_shot_acak":
                        contoh = bank.acak(peta[t], arah)
                    else:
                        contoh = bank.retrieval(t, arah)
                    prompts.append(buat_prompt(t, arah, contoh))
                return terjemah_llm(prompts, tok, model, c_llm["token_baru_maks"])

            if not contoh_dicetak and kd != "zero_shot":
                sumber, _ = sumber_dan_rujukan(df, a)
                print("Contoh prompt:\n" + "-" * 60)
                print(buat_prompt(sumber.iat[0], a, bank.retrieval(sumber.iat[0], a)))
                print("-" * 60)
                contoh_dicetak = True
            jalankan_unit(sistem, s, a, {**param, "kondisi": kd}, dengan_belah_dua(fungsi), c_llm["ukuran_batch"])
        catatan[nama_model] = "lengkap"
        del model, tok
        bersihkan_gpu()
    tulis_atomik(P_S09, json.dumps(catatan, indent=2, ensure_ascii=False))
    tahap.selesai({"catatan": catatan})

---
# Bagian D. Evaluasi otomatis dan analisis

## S10. Evaluasi otomatis semua sistem

**Tujuan:** menghitung skor setiap sistem pada setiap set uji, arah, dan subset.

**Yang dilakukan sel ini:**

1. Membaca semua berkas di `prediksi/`. Sistem yang prediksinya belum lengkap untuk suatu subset dilewati dan dicatat.
2. Menghitung **chrF++** (metrik utama) dan **BLEU** dengan sacreBLEU pada tingkat korpus untuk setiap kombinasi berikut:
   - set uji `kami`: semua, per arah asli, per genre, dan per status validasi manusia;
   - set uji `nusatranslation`: semua dan per label varietas rujukan (`AM`, `Toba_saja`, `tak_bertanda`);
   - set uji `nusax` dan `nusatranslation_toba`: semua.
3. Menghitung **interval kepercayaan 95%** chrF++ dengan *bootstrap resampling* (pengambilan sampel ulang) sebanyak 1.000 kali, dan **uji beda berpasangan** (*paired bootstrap*) untuk pasangan sistem di `PERBANDINGAN`. Sampel ulangnya sama untuk semua sistem, sehingga perbedaannya adil.
4. Merata-ratakan *run* yang diulang dengan beberapa benih (rata-rata dan simpangan baku).
5. Menghitung **persentase keluaran bertanda AM** pada arah id→bbc, untuk melihat apakah model menulis varietas lain (temuan naskah sebelumnya).
6. Menghitung skor tingkat kalimat pada set uji `kami`, untuk korelasi dengan penilaian manusia di S13.
7. (Opsional) COMET, bila `hitung_comet=True`. COMET hanya bahan analisis, karena tidak mendukung Batak.

**Keluaran yang diharapkan:** `s10_skor.csv` (tabel lengkap), `s10_perbandingan.csv`, `s10_rata_benih.csv`, `s10_varietas_keluaran.csv`, `s10_skor_kalimat.csv`, dan `s10_unit_belum_lengkap.csv`.

In [ ]:
# S10 Evaluasi otomatis
from sacrebleu.metrics import CHRF, BLEU

CHRF_M = CHRF(word_order=2)     # chrF++
BLEU_M = BLEU()
# Pasangan sistem yang diuji bedanya (A - B). Pasangan LLM ditambahkan otomatis.
PERBANDINGAN = [
    ("ft_kami+nusatranslation_s42", "ft_nusatranslation_s42"),
    ("ft_kami+nusax_s42", "ft_nusax_s42"),
    ("ft_kami_s42", "ft_nusax_s42"),
    ("ft_kami_s42", "ft_nusatranslation_s42"),
]
P_S10 = DIR_KELUARAN / "s10_skor.csv"
P_S10_BEDA = DIR_KELUARAN / "s10_perbandingan.csv"
P_S10_BENIH = DIR_KELUARAN / "s10_rata_benih.csv"
P_S10_VAR = DIR_KELUARAN / "s10_varietas_keluaran.csv"
P_S10_KALIMAT = DIR_KELUARAN / "s10_skor_kalimat.csv"
P_S10_KURANG = DIR_KELUARAN / "s10_unit_belum_lengkap.csv"


def daftar_berkas_prediksi():
    return sorted(p for p in DIR_PREDIKSI.glob("*/*.jsonl") if not p.name.endswith(".lama.jsonl"))


def muat_semua_prediksi():
    pred = {}
    for p in daftar_berkas_prediksi():
        sistem, arah = p.parent.name, p.stem.split(".")[-1]
        baris = p.read_text(encoding="utf-8").splitlines()[1:]
        d = pred.setdefault(sistem, {})
        for b in baris:
            try:
                r = json.loads(b)
                d[(arah, r["uji_id"])] = r["hipotesis"]
            except Exception:
                pass
    return pred


def daftar_subset(uji):
    hasil = []
    for s in ["kami", "nusax", "nusatranslation", "nusatranslation_toba"]:
        d = uji[uji.set_uji == s].reset_index(drop=True)
        if d.empty:
            continue
        hasil.append((s, "semua", "semua", d))
        dimensi = {"kami": ["arah_asli", "genre", "validasi_manusia"], "nusatranslation": ["varietas_rujukan"]}.get(s, [])
        for dim in dimensi:
            for nilai, g in d.groupby(dim):
                if str(nilai).strip():
                    hasil.append((s, dim, str(nilai), g.reset_index(drop=True)))
    return hasil


def statistik_kalimat(metrik, hip, ruj):
    return np.array(metrik._extract_corpus_statistics(hip, [ruj]), dtype=float)


def skor_dari_statistik(metrik, S):
    return metrik._compute_score_from_stats(S.sum(0).tolist()).score


_CACHE_INDEKS = {}


def indeks_bootstrap(n, B):
    # Indeks sampel ulang yang sama untuk semua sistem dengan n kalimat yang sama.
    if (n, B) not in _CACHE_INDEKS:
        _CACHE_INDEKS[(n, B)] = np.random.RandomState(12345).randint(0, n, size=(B, n))
    return _CACHE_INDEKS[(n, B)]


berkas_pred = daftar_berkas_prediksi()
c_eval = KONFIGURASI["evaluasi"]
tahap = Tahap("S10", "Evaluasi otomatis", [P_S05_UJI] + berkas_pred,
              [P_S10, P_S10_BEDA, P_S10_BENIH, P_S10_VAR, P_S10_KALIMAT, P_S10_KURANG],
              {"evaluasi": c_eval, "perbandingan": PERBANDINGAN})
if not tahap.bisa_dilewati():
    uji = baca_csv(P_S05_UJI)
    pred = muat_semua_prediksi()
    subset = daftar_subset(uji)
    ada_berkas = {(p.parent.name, p.stem.split(".")[0], p.stem.split(".")[-1]) for p in berkas_pred}
    B = c_eval["n_bootstrap"]
    baris, kurang, stat_simpan = [], [], {}
    for sistem in sorted(pred):
        for s, dim, nilai, d in subset:
            for arah in ARAH:
                hip = [pred[sistem].get((arah, i)) for i in d["uji_id"]]
                if any(h is None for h in hip):
                    if dim == "semua" and (sistem, s, arah) in ada_berkas:
                        kurang.append({"sistem": sistem, "set_uji": s, "arah": arah,
                                       "terisi": sum(h is not None for h in hip), "total": len(hip)})
                    continue
                _, ruj = sumber_dan_rujukan(d, arah)
                ruj = list(ruj)
                Sc, Sb = statistik_kalimat(CHRF_M, hip, ruj), statistik_kalimat(BLEU_M, hip, ruj)
                r = {"sistem": sistem, "set_uji": s, "dimensi": dim, "nilai": nilai, "arah": arah, "n": len(hip),
                     "chrf": round(skor_dari_statistik(CHRF_M, Sc), 2), "bleu": round(skor_dari_statistik(BLEU_M, Sb), 2)}
                if dim in ("semua", "arah_asli"):
                    idx = indeks_bootstrap(len(hip), B)
                    sampel = np.array([skor_dari_statistik(CHRF_M, Sc[ix]) for ix in idx])
                    r["chrf_ik_bawah"], r["chrf_ik_atas"] = np.round(np.percentile(sampel, [2.5, 97.5]), 2)
                    stat_simpan[(sistem, s, dim, nilai, arah)] = (Sc, sampel)
                baris.append(r)
    skor = pd.DataFrame(baris)
    simpan_csv(skor, P_S10)
    simpan_csv(pd.DataFrame(kurang, columns=["sistem", "set_uji", "arah", "terisi", "total"]), P_S10_KURANG)
    if kurang:
        print(f"PERINGATAN: {len(kurang)} unit belum lengkap dan tidak dinilai (lihat s10_unit_belum_lengkap.csv).")

    # --- uji beda berpasangan ---
    pasangan = list(PERBANDINGAN)
    for sistem in pred:
        if sistem.startswith("llm_") and sistem.endswith("_few_shot_retrieval"):
            dasar = sistem[:-len("few_shot_retrieval")]
            pasangan += [(sistem, dasar + "few_shot_acak"), (dasar + "few_shot_acak", dasar + "zero_shot")]
    beda = []
    for a, b in pasangan:
        for (sis, s, dim, nilai, arah), (Sa, sampel_a) in stat_simpan.items():
            if sis != a or (b, s, dim, nilai, arah) not in stat_simpan:
                continue
            Sb_, sampel_b = stat_simpan[(b, s, dim, nilai, arah)]
            d = sampel_a - sampel_b
            selisih = skor_dari_statistik(CHRF_M, Sa) - skor_dari_statistik(CHRF_M, Sb_)
            p = 2 * min((d <= 0).mean(), (d >= 0).mean())
            beda.append({"sistem_a": a, "sistem_b": b, "set_uji": s, "dimensi": dim, "nilai": nilai, "arah": arah,
                         "selisih_chrf": round(selisih, 2), "ik_bawah": round(np.percentile(d, 2.5), 2),
                         "ik_atas": round(np.percentile(d, 97.5), 2), "p_dua_sisi": round(min(1.0, p), 4)})
    beda = pd.DataFrame(beda)
    simpan_csv(beda, P_S10_BEDA)

    # --- rata-rata antar-benih ---
    s_semua = skor[skor.dimensi.isin(["semua", "arah_asli"])].copy()
    s_semua["grup"] = s_semua["sistem"].str.replace(r"_s\d+$", "", regex=True)
    benih = (s_semua[s_semua.sistem.str.match(r"^ft_.*_s\d+$")]
             .groupby(["grup", "set_uji", "dimensi", "nilai", "arah"])
             .agg(n_benih=("sistem", "nunique"), chrf_rata=("chrf", "mean"), chrf_sb=("chrf", "std"),
                  bleu_rata=("bleu", "mean"), bleu_sb=("bleu", "std")).round(2).reset_index())
    simpan_csv(benih, P_S10_BENIH)

    # --- varietas keluaran id->bbc ---
    var = []
    for s, dim, nilai, d in subset:
        if dim != "semua":
            continue
        kandidat = [("rujukan", list(d["teks_bbc"]))]
        for sistem in sorted(pred):
            hip = [pred[sistem].get(("id2bbc", i)) for i in d["uji_id"]]
            if all(h is not None for h in hip):
                kandidat.append((sistem, hip))
        for sistem, teks in kandidat:
            lab = pd.Series([label_varietas(t) for t in teks])
            var.append({"sistem": sistem, "set_uji": s, "n": len(teks),
                        "persen_AM": round((lab == "AM").mean() * 100, 2),
                        "persen_Toba_saja": round((lab == "Toba_saja").mean() * 100, 2)})
    var = pd.DataFrame(var)
    simpan_csv(var, P_S10_VAR)

    # --- skor tingkat kalimat pada set uji kami (untuk S13) ---
    d = uji[uji.set_uji == "kami"]
    kal = []
    for sistem in sorted(pred):
        for arah in ARAH:
            _, ruj = sumber_dan_rujukan(d, arah)
            for i, r in zip(d["uji_id"], ruj):
                h = pred[sistem].get((arah, i))
                if h is not None:
                    kal.append({"sistem": sistem, "arah": arah, "uji_id": i,
                                "chrf": round(CHRF_M.sentence_score(h, [r]).score, 3),
                                "bleu": round(sacrebleu.sentence_bleu(h, [r]).score, 3)})
    simpan_csv(pd.DataFrame(kal), P_S10_KALIMAT)

    # --- COMET opsional ---
    if c_eval["hitung_comet"]:
        try:
            pasang_bila_perlu("comet", "unbabel-comet")
            from comet import download_model, load_from_checkpoint
            model_comet = load_from_checkpoint(download_model(c_eval["model_comet"]))
            nilai_comet = {}
            for sistem in sorted(pred):
                for s, dim, nilai, dd in subset:
                    if dim != "semua":
                        continue
                    for arah in ARAH:
                        hip = [pred[sistem].get((arah, i)) for i in dd["uji_id"]]
                        if any(h is None for h in hip):
                            continue
                        src, ruj = sumber_dan_rujukan(dd, arah)
                        data = [{"src": a, "mt": h, "ref": r} for a, h, r in zip(src, hip, ruj)]
                        nilai_comet[(sistem, s, arah)] = model_comet.predict(data, batch_size=64, gpus=1 if PERANGKAT == "cuda" else 0).system_score
            skor["comet"] = [round(nilai_comet.get((r.sistem, r.set_uji, r.arah), np.nan), 4) if r.dimensi == "semua" else np.nan
                             for r in skor.itertuples()]
            simpan_csv(skor, P_S10)
        except Exception as e:
            print("COMET dilewati:", type(e).__name__, e)

    utama = skor[(skor.dimensi == "semua")].pivot_table(index="sistem", columns=["set_uji", "arah"], values="chrf")
    print("chrF++ per sistem (set uji lengkap):")
    tampilkan(utama.round(1), 60)
    tahap.selesai({"sistem": skor["sistem"].nunique(), "baris_skor": len(skor), "unit_belum_lengkap": len(kurang),
                   "perbandingan": len(beda)})

## S11. Analisis per arah asli, genre, dan varietas keluaran (RQ2)

**Tujuan:** menyusun tabel dan gambar untuk naskah.

**Yang dilakukan sel ini:**

1. **Tabel utama:** chrF++ setiap sistem pada setiap set uji dan arah, lengkap dengan interval kepercayaan.
2. **Analisis arah asli (RQ2):** chrF++ pada set uji `kami` untuk subset `asli_toba` dan `asli_indonesia`, beserta selisihnya. Selisih positif berarti sistem lebih baik pada teks yang aslinya berbahasa Batak Toba.
3. **Analisis genre.**
4. **Gambar:** chrF++ per arah asli untuk sistem terpilih, dan persentase keluaran bertanda AM.

**Penting saat menafsirkan RQ2:** arah asli dan genre saling berkaitan. Dokumen asli Toba sebagian besar berupa sastra dan cerita rakyat, sedangkan dokumen asli Indonesia sebagian besar berupa media dan edukasi. Selisih antar-arah asli bisa saja berasal dari perbedaan genre. Sel ini mencetak tabulasi silang arah × genre pada set uji supaya keterkaitan itu terlihat dan bisa dibahas sebagai keterbatasan.

**Keluaran yang diharapkan:** `s11_tabel_utama.csv`, `s11_arah_asli.csv`, `s11_genre.csv`, `gambar/s11_arah_asli.png`, dan `gambar/s11_varietas_keluaran.png`.

In [ ]:
# S11 Analisis
P_S11_UTAMA = DIR_KELUARAN / "s11_tabel_utama.csv"
P_S11_ARAH = DIR_KELUARAN / "s11_arah_asli.csv"
P_S11_GENRE = DIR_KELUARAN / "s11_genre.csv"
P_S11_G_ARAH = DIR_GAMBAR / "s11_arah_asli.png"
P_S11_G_VAR = DIR_GAMBAR / "s11_varietas_keluaran.png"


def urutan_sistem(nama):
    for i, awal in enumerate(["copy_source", "nllb", "madlad", "ft_", "llm_"]):
        if nama.startswith(awal):
            return (i, nama)
    return (9, nama)


tahap = Tahap("S11", "Analisis", [P_S10, P_S10_VAR, P_S05_UJI], [P_S11_UTAMA, P_S11_ARAH, P_S11_GENRE, P_S11_G_ARAH, P_S11_G_VAR], {})
if not tahap.bisa_dilewati():
    skor = pd.read_csv(P_S10)
    uji = baca_csv(P_S05_UJI)
    print("Tabulasi silang arah asli x genre pada set uji kami:")
    print(pd.crosstab(uji[uji.set_uji == "kami"]["genre"], uji[uji.set_uji == "kami"]["arah_asli"]))

    # 1) Tabel utama dengan IK 95%
    u = skor[skor.dimensi == "semua"].copy()
    u["teks"] = u.apply(lambda r: f"{r.chrf:.1f} [{r.chrf_ik_bawah:.1f}, {r.chrf_ik_atas:.1f}]"
                        if pd.notna(r.get("chrf_ik_bawah")) else f"{r.chrf:.1f}", axis=1)
    utama = u.pivot_table(index="sistem", columns=["set_uji", "arah"], values="teks", aggfunc="first")
    utama = utama.loc[sorted(utama.index, key=urutan_sistem)]
    utama.columns = [f"{a}|{b}" for a, b in utama.columns]
    simpan_csv(utama.reset_index(), P_S11_UTAMA)
    tampilkan(utama, 60)

    # 2) Arah asli
    a = skor[(skor.set_uji == "kami") & (skor.dimensi == "arah_asli")]
    arah = a.pivot_table(index="sistem", columns=["arah", "nilai"], values="chrf")
    for r in ARAH:
        if (r, "asli_toba") in arah.columns and (r, "asli_indonesia") in arah.columns:
            arah[(r, "selisih_toba_minus_indonesia")] = arah[(r, "asli_toba")] - arah[(r, "asli_indonesia")]
    arah = arah.sort_index(axis=1).round(2)
    arah = arah.loc[sorted(arah.index, key=urutan_sistem)]
    arah_datar = arah.copy()
    arah_datar.columns = [f"{x}|{y}" for x, y in arah_datar.columns]
    simpan_csv(arah_datar.reset_index(), P_S11_ARAH)
    tampilkan(arah_datar, 60)

    # 3) Genre
    g = skor[(skor.set_uji == "kami") & (skor.dimensi == "genre")]
    genre = g.pivot_table(index="sistem", columns=["arah", "nilai"], values="chrf").round(2)
    genre = genre.loc[sorted(genre.index, key=urutan_sistem)]
    genre.columns = [f"{x}|{y}" for x, y in genre.columns]
    simpan_csv(genre.reset_index(), P_S11_GENRE)

    # 4a) Gambar arah asli: sistem terpilih (copy-source, zero-shot terbaik, run ft benih 42, LLM terbaik)
    kandidat = [s for s in arah.index if s == "copy_source" or s.endswith("_s42")]
    for awal in ["nllb", "madlad", "llm_"]:
        pilihan = [s for s in arah.index if s.startswith(awal)]
        if pilihan and ("id2bbc", "asli_toba") in arah.columns:
            kandidat.append(max(pilihan, key=lambda s: np.nan_to_num(arah.loc[s].get(("id2bbc", "asli_indonesia"), 0))))
    kandidat = sorted(set(kandidat), key=urutan_sistem)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
    for ax, r in zip(axes, ARAH):
        kol = [k for k in [(r, "asli_toba"), (r, "asli_indonesia")] if k in arah.columns]
        if kandidat and kol:
            arah.loc[kandidat, kol].plot(kind="bar", ax=ax, color=["#3B6FD8", "#D98A1C"][:len(kol)], rot=60)
            ax.legend([k[1] for k in kol])
        ax.set_title({"id2bbc": "Indonesia -> Batak Toba", "bbc2id": "Batak Toba -> Indonesia"}[r])
        ax.set_ylabel("chrF++")
        ax.set_xlabel("")
    plt.tight_layout()
    plt.savefig(P_S11_G_ARAH, dpi=200)
    plt.show()

    # 4b) Gambar varietas keluaran (id->bbc)
    var = pd.read_csv(P_S10_VAR)
    v = var[var.set_uji.isin(["kami", "nusatranslation"])].pivot_table(index="sistem", columns="set_uji", values="persen_AM")
    v = v.loc[sorted(v.index, key=lambda s: (s != "rujukan", urutan_sistem(s)))]
    ax = v.plot(kind="bar", figsize=(12, 4.5), color=["#2E8B57", "#D98A1C"][:v.shape[1]], rot=70)
    ax.set_ylabel("% keluaran bertanda AM")
    ax.set_xlabel("")
    ax.set_title("Keluaran id->bbc bertanda Angkola-Mandailing")
    plt.tight_layout()
    plt.savefig(P_S11_G_VAR, dpi=200)
    plt.show()
    tahap.selesai({"sistem_di_tabel_utama": len(utama)})

---
# Bagian E. Evaluasi manusia

## S12. Siapkan lembar ESA untuk dua penilai (RQ6)

**Tujuan:** menyiapkan berkas Excel yang siap dikirim ke **dua penilai** penutur asli Batak Toba yang fasih berbahasa Indonesia. Dua penilai yang menilai secara terpisah memungkinkan kita menghitung kesepakatan antarpenilai. Ini menjawab kelemahan naskah sebelumnya, yang hanya memakai satu penilai.

**Yang dilakukan sel ini:**

1. Mengambil `n_kalimat` kalimat dari set uji `kami`, berstrata arah asli (separuh asli Toba bila cukup).
2. Mengambil keluaran sistem yang ada di `esa.sistem`. `LLM_TERBAIK` diganti otomatis dengan kondisi LLM yang chrF++-nya tertinggi pada set uji `kami` arah id→bbc.
3. Membuat satu berkas per penilai (`esa_penilai_A.xlsx`, `esa_penilai_B.xlsx`) dengan lembar petunjuk, lembar `id2bbc`, dan lembar `bbc2id`. Urutan butir diacak berbeda untuk setiap penilai, dan nama sistem disembunyikan.
4. Menyimpan kunci jawaban (`kunci_esa.csv`, berisi butir → sistem). **Jangan kirim berkas kunci ke penilai.**

**Keluaran yang diharapkan:** tiga berkas di `evaluasi_manusia/`. Dengan pengaturan bawaan, ada 60 kalimat × 4 sistem × 2 arah = 480 butir per penilai, atau kira-kira 6 sampai 8 jam kerja per penilai.

**Langkah setelah sel ini:** kirim `esa_penilai_A.xlsx` dan `esa_penilai_B.xlsx` ke penilai. Setelah diisi, unggah kembali ke `evaluasi_manusia/terisi/` dengan nama yang sama, lalu jalankan S13.

In [ ]:
# S12 Lembar ESA
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE


def aman_excel(teks):
    # Karakter kontrol dari keluaran model tidak boleh masuk ke sel Excel.
    return ILLEGAL_CHARACTERS_RE.sub("", str(teks))


P_KUNCI_ESA = DIR_ESA / "kunci_esa.csv"
P_LEMBAR_ESA = {p: DIR_ESA / f"esa_penilai_{p}.xlsx" for p in KONFIGURASI["esa"]["penilai"]}
PETUNJUK_ESA = [
    "Petunjuk penilaian terjemahan (Error Span Annotation)",
    "",
    "Anda akan menilai terjemahan antara Bahasa Indonesia dan Bahasa Batak Toba. Lembar 'id2bbc' berisi terjemahan dari Indonesia ke Batak Toba;",
    "lembar 'bbc2id' berisi terjemahan dari Batak Toba ke Indonesia. Urutan sudah diacak dan nama sistem disembunyikan.",
    "",
    "Untuk setiap baris:",
    "1. Baca kalimat sumber, lalu baca terjemahannya.",
    "2. Salin bagian terjemahan yang keliru ke kolom kesalahan_mayor atau kesalahan_minor. Pisahkan beberapa bagian dengan tanda ' | '.",
    "   Kesalahan mayor: makna berubah, hilang, atau tidak dapat dipahami.",
    "   Kesalahan minor: makna tetap tersampaikan, tetapi ada kekeliruan ejaan, tata bahasa, atau pilihan kata yang kurang wajar.",
    "3. Khusus lembar id2bbc: isi kolom variety dengan 'ya' bila terjemahan memakai kata yang benar maknanya tetapi bukan Batak Toba",
    "   (misalnya Angkola, Mandailing, Simalungun, Banjar, atau Bahasa Indonesia). Catat juga kata itu sebagai kesalahan minor.",
    "   Isi 'tidak' bila seluruh kata adalah Batak Toba.",
    "4. Beri skor 0 sampai 100 di kolom skor. 0 berarti tidak ada makna yang tersampaikan dalam bahasa sasaran; 100 berarti sempurna.",
    "   Terjemahan yang tidak diterjemahkan (masih dalam bahasa sumber) dinilai dari makna yang tersampaikan dalam bahasa sasaran.",
    "5. Kolom catatan boleh diisi bila perlu.",
    "",
    "Nilai setiap baris secara mandiri dan jangan berdiskusi dengan penilai lain. Jangan mengubah kolom item_id, sumber, dan terjemahan.",
]

tahap = Tahap("S12", "Lembar ESA", [P_S05_UJI, P_S10], [P_KUNCI_ESA] + list(P_LEMBAR_ESA.values()),
              {"esa": KONFIGURASI["esa"], "benih": KONFIGURASI["benih_global"]})
if not tahap.bisa_dilewati():
    c_esa = KONFIGURASI["esa"]
    skor = pd.read_csv(P_S10)
    pred = muat_semua_prediksi()
    sistem_esa = []
    for s in c_esa["sistem"]:
        if s == "LLM_TERBAIK":
            k = skor[(skor.set_uji == "kami") & (skor.dimensi == "semua") & (skor.arah == "id2bbc")
                     & skor.sistem.str.startswith("llm_")]
            if k.empty:
                print("Belum ada sistem LLM; LLM_TERBAIK dilewati.")
                continue
            s = k.sort_values("chrf").iloc[-1]["sistem"]
            print("LLM_TERBAIK =", s)
        if s not in pred:
            print(f"PERINGATAN: sistem {s} tidak punya prediksi; dilewati.")
            continue
        sistem_esa.append(s)
    assert len(sistem_esa) >= 2, "Minimal dua sistem dibutuhkan untuk ESA."

    uji_kami = muat_set_uji("kami")
    lengkap = uji_kami["uji_id"].map(lambda i: all((a, i) in pred[s] for s in sistem_esa for a in ARAH))
    uji_kami = uji_kami[lengkap]
    rng = np.random.RandomState(KONFIGURASI["benih_global"])
    n = min(c_esa["n_kalimat"], len(uji_kami))
    toba = uji_kami[uji_kami.arah_asli == "asli_toba"]
    n_toba = min(len(toba), n // 2)
    pilih = pd.concat([toba.sample(n=n_toba, random_state=rng),
                       uji_kami[uji_kami.arah_asli != "asli_toba"].sample(n=n - n_toba, random_state=rng)])
    print(f"Kalimat ESA: {len(pilih)} ({n_toba} asli Toba). Sistem: {sistem_esa}")

    butir = []
    for arah in ARAH:
        sumber, _ = sumber_dan_rujukan(pilih, arah)
        for i, src in zip(pilih["uji_id"], sumber):
            for s in sistem_esa:
                butir.append({"arah": arah, "uji_id": i, "sistem": s, "sumber": src, "terjemahan": pred[s][(arah, i)]})
    butir = pd.DataFrame(butir)
    butir["item_id"] = [f"esa-{k:04d}" for k in rng.permutation(len(butir))]
    simpan_csv(butir[["item_id", "arah", "uji_id", "sistem"]], P_KUNCI_ESA)

    for j, penilai in enumerate(c_esa["penilai"]):
        wb = Workbook()
        ws = wb.active
        ws.title = "Petunjuk"
        for r, t in enumerate(PETUNJUK_ESA, 1):
            ws.cell(row=r, column=1, value=t)
        ws["A1"].font = Font(bold=True, size=14)
        ws.column_dimensions["A"].width = 150
        for arah in ARAH:
            d = butir[butir.arah == arah].sample(frac=1, random_state=KONFIGURASI["benih_global"] + 100 * (j + 1))
            lembar = wb.create_sheet(arah)
            kolom = ["item_id", "sumber", "terjemahan", "kesalahan_mayor", "kesalahan_minor"] + \
                    (["variety"] if arah == "id2bbc" else []) + ["skor", "catatan"]
            lembar.append(kolom)
            for r in d.itertuples():
                lembar.append([r.item_id, aman_excel(r.sumber), aman_excel(r.terjemahan)] + [""] * (len(kolom) - 3))
            lebar = {"item_id": 11, "sumber": 55, "terjemahan": 55, "kesalahan_mayor": 30, "kesalahan_minor": 30,
                     "variety": 9, "skor": 8, "catatan": 30}
            for k, nama in enumerate(kolom, 1):
                huruf = lembar.cell(row=1, column=k).column_letter
                lembar.column_dimensions[huruf].width = lebar[nama]
                lembar.cell(row=1, column=k).font = Font(bold=True)
                for rr in range(2, len(d) + 2):
                    sel = lembar.cell(row=rr, column=k)
                    sel.alignment = Alignment(wrap_text=True, vertical="top")
                    if nama not in ("item_id", "sumber", "terjemahan"):
                        sel.fill = PatternFill("solid", fgColor="FFF2CC")
                if nama == "skor":
                    dv = DataValidation(type="whole", operator="between", formula1="0", formula2="100", allow_blank=True)
                    dv.add(f"{huruf}2:{huruf}{len(d) + 1}")
                    lembar.add_data_validation(dv)
                if nama == "variety":
                    dv = DataValidation(type="list", formula1='"ya,tidak"', allow_blank=True)
                    dv.add(f"{huruf}2:{huruf}{len(d) + 1}")
                    lembar.add_data_validation(dv)
            lembar.freeze_panes = "B2"
        sementara = P_LEMBAR_ESA[penilai].with_suffix(".tmp.xlsx")
        wb.save(sementara)
        os.replace(sementara, P_LEMBAR_ESA[penilai])
    tahap.selesai({"kalimat": len(pilih), "sistem": sistem_esa, "butir_per_penilai": len(butir)})

## S13. Analisis ESA dan kesepakatan antarpenilai (RQ6)

**Tujuan:** merangkum penilaian manusia dan memeriksa apakah metrik otomatis sejalan dengannya.

**Prasyarat:** berkas `esa_penilai_A.xlsx` dan `esa_penilai_B.xlsx` yang sudah diisi ada di `evaluasi_manusia/terisi/`. Kalau belum ada, sel ini hanya mencetak pesan dan tidak menghentikan notebook.

**Yang dihitung:**

1. **Skor manusia per sistem dan arah:** rata-rata skor kedua penilai, dengan interval kepercayaan 95% dari *bootstrap* atas kalimat. Perbedaan antarsistem diuji dengan *Wilcoxon signed-rank test* (uji peringkat bertanda Wilcoxon).
2. **Kesepakatan antarpenilai:**
   - skor 0 sampai 100: korelasi Pearson, korelasi Spearman, dan **Krippendorff's alpha** (alfa Krippendorff) skala interval;
   - ada atau tidaknya kesalahan mayor, serta tag *variety* (khusus id→bbc): **Cohen's kappa** (kappa Cohen).
3. **Korelasi metrik dan manusia:** **Kendall's tau-b** (tau-b Kendall) antara chrF++ atau BLEU tingkat kalimat dan skor manusia rata-rata, dengan interval kepercayaan dari *bootstrap* atas kalimat. Juga dihitung khusus untuk pasangan sistem *fine-tuned* (sistem `ft_`), karena naskah sebelumnya menemukan korelasi yang lemah di sana.
4. **Tag *variety* per sistem:** persentase butir id→bbc yang ditandai memakai kata non-Toba.

**Keluaran yang diharapkan:** `s13_skor_manusia.csv`, `s13_wilcoxon.csv`, `s13_kesepakatan.csv`, `s13_korelasi_metrik.csv`, dan `s13_variety.csv`.

In [ ]:
# S13 Analisis ESA
from scipy import stats
from sklearn.metrics import cohen_kappa_score

P_ESA_TERISI = {p: DIR_ESA / "terisi" / f"esa_penilai_{p}.xlsx" for p in KONFIGURASI["esa"]["penilai"]}
P_S13 = {n: DIR_KELUARAN / f"s13_{n}.csv" for n in ["skor_manusia", "wilcoxon", "kesepakatan", "korelasi_metrik", "variety"]}


def krippendorff_alpha_interval(x1, x2):
    # Alfa Krippendorff skala interval untuk dua penilai tanpa data hilang.
    v = np.concatenate([x1, x2]).astype(float)
    n = len(v)
    ss = ((v - v.mean()) ** 2).sum()
    if ss == 0:
        return float("nan")
    return 1 - (n - 1) * ((np.asarray(x1, float) - np.asarray(x2, float)) ** 2).sum() / (n * ss)


def baca_lembar_terisi(path, penilai):
    bagian = []
    for arah in ARAH:
        d = pd.read_excel(path, sheet_name=arah, dtype=str).fillna("")
        d["arah"] = arah
        bagian.append(d)
    d = pd.concat(bagian, ignore_index=True)
    d["skor"] = pd.to_numeric(d["skor"], errors="coerce")
    salah = d[d["skor"].isna() | (d["skor"] < 0) | (d["skor"] > 100)]
    if len(salah):
        raise ValueError(f"Penilai {penilai}: {len(salah)} butir belum diberi skor 0-100, misalnya {list(salah['item_id'][:5])}")
    d["mayor"] = (d["kesalahan_mayor"].str.strip() != "").astype(int)
    d["variety"] = d.get("variety", "").astype(str).str.strip().str.lower().eq("ya").astype(int)
    return d.rename(columns={c: f"{c}_{penilai}" for c in ["skor", "mayor", "variety", "kesalahan_mayor", "kesalahan_minor", "catatan"]})


def bootstrap_rata(nilai_per_kalimat, B=1000, benih=12345):
    rng = np.random.RandomState(benih)
    x = np.asarray(nilai_per_kalimat, float)
    sampel = x[rng.randint(0, len(x), size=(B, len(x)))].mean(1)
    return np.percentile(sampel, [2.5, 97.5])


if not all(p.exists() for p in P_ESA_TERISI.values()):
    print("S13 dilewati: lembar ESA terisi belum ada. Unggah berkas berikut lalu jalankan sel ini lagi:")
    for p in P_ESA_TERISI.values():
        print("  ", p, "(ada)" if p.exists() else "(belum ada)")
else:
    tahap = Tahap("S13", "Analisis ESA", [P_KUNCI_ESA, P_S10_KALIMAT] + list(P_ESA_TERISI.values()), list(P_S13.values()), {})
    if not tahap.bisa_dilewati():
        A, Bp = KONFIGURASI["esa"]["penilai"][:2]
        kunci = baca_csv(P_KUNCI_ESA)
        d = kunci.merge(baca_lembar_terisi(P_ESA_TERISI[A], A)[["item_id", f"skor_{A}", f"mayor_{A}", f"variety_{A}"]], on="item_id")
        d = d.merge(baca_lembar_terisi(P_ESA_TERISI[Bp], Bp)[["item_id", f"skor_{Bp}", f"mayor_{Bp}", f"variety_{Bp}"]], on="item_id")
        assert len(d) == len(kunci), "Ada butir yang hilang dari lembar terisi"
        d["skor"] = d[[f"skor_{A}", f"skor_{Bp}"]].mean(1)

        # 1) Skor per sistem
        baris = []
        for (arah, sistem), g in d.groupby(["arah", "sistem"]):
            ik = bootstrap_rata(g["skor"])
            baris.append({"arah": arah, "sistem": sistem, "n": len(g), "skor_rata": round(g["skor"].mean(), 2),
                          f"skor_{A}": round(g[f"skor_{A}"].mean(), 2), f"skor_{Bp}": round(g[f"skor_{Bp}"].mean(), 2),
                          "ik_bawah": round(ik[0], 2), "ik_atas": round(ik[1], 2),
                          "persen_mayor": round(g[[f"mayor_{A}", f"mayor_{Bp}"]].mean(1).mean() * 100, 1)})
        skor_manusia = pd.DataFrame(baris)
        simpan_csv(skor_manusia, P_S13["skor_manusia"])
        tampilkan(skor_manusia, 30)

        wil = []
        for arah, g in d.groupby("arah"):
            lebar = g.pivot_table(index="uji_id", columns="sistem", values="skor")
            sis = list(lebar.columns)
            for i in range(len(sis)):
                for j in range(i + 1, len(sis)):
                    x, y = lebar[sis[i]], lebar[sis[j]]
                    try:
                        p = stats.wilcoxon(x, y).pvalue
                    except ValueError:
                        p = float("nan")
                    wil.append({"arah": arah, "sistem_a": sis[i], "sistem_b": sis[j],
                                "selisih": round((x - y).mean(), 2), "p_wilcoxon": p})
        simpan_csv(pd.DataFrame(wil), P_S13["wilcoxon"])

        # 2) Kesepakatan antarpenilai
        kes = []
        for arah, g in d.groupby("arah"):
            x1, x2 = g[f"skor_{A}"].values, g[f"skor_{Bp}"].values
            r = {"arah": arah, "n": len(g), "pearson": round(stats.pearsonr(x1, x2)[0], 3),
                 "spearman": round(stats.spearmanr(x1, x2)[0], 3),
                 "krippendorff_alpha_interval": round(krippendorff_alpha_interval(x1, x2), 3),
                 "kappa_mayor": round(cohen_kappa_score(g[f"mayor_{A}"], g[f"mayor_{Bp}"]), 3)}
            if arah == "id2bbc":
                r["kappa_variety"] = round(cohen_kappa_score(g[f"variety_{A}"], g[f"variety_{Bp}"]), 3)
            kes.append(r)
        kes = pd.DataFrame(kes)
        simpan_csv(kes, P_S13["kesepakatan"])
        tampilkan(kes)

        # 3) Korelasi metrik dan manusia (Kendall tau-b, bootstrap atas kalimat)
        kal = baca_csv(P_S10_KALIMAT)
        for k in ["chrf", "bleu"]:
            kal[k] = kal[k].astype(float)
        dk = d.merge(kal, on=["sistem", "arah", "uji_id"], how="left")
        kor = []
        rng = np.random.RandomState(12345)
        for arah, g0 in dk.groupby("arah"):
            for cakupan, g in [("semua_sistem", g0), ("hanya_ft", g0[g0.sistem.str.startswith("ft_")])]:
                if g["sistem"].nunique() < 2:
                    continue
                g = g.reset_index(drop=True)
                indeks_per_kalimat = list(g.groupby("uji_id").indices.values())
                for metrik in ["chrf", "bleu"]:
                    x, y = g[metrik].values, g["skor"].values
                    tau = stats.kendalltau(x, y).statistic
                    sampel = []
                    for _ in range(1000):
                        pilih = rng.randint(0, len(indeks_per_kalimat), size=len(indeks_per_kalimat))
                        ix = np.concatenate([indeks_per_kalimat[k] for k in pilih])
                        sampel.append(stats.kendalltau(x[ix], y[ix]).statistic)
                    kor.append({"arah": arah, "cakupan": cakupan, "metrik": metrik, "n_butir": len(g),
                                "kendall_tau_b": round(tau, 3), "ik_bawah": round(np.nanpercentile(sampel, 2.5), 3),
                                "ik_atas": round(np.nanpercentile(sampel, 97.5), 3)})
        kor = pd.DataFrame(kor)
        simpan_csv(kor, P_S13["korelasi_metrik"])
        tampilkan(kor, 20)

        # 4) Tag variety per sistem (id2bbc)
        v = d[d.arah == "id2bbc"].groupby("sistem")[[f"variety_{A}", f"variety_{Bp}"]].mean().mul(100).round(1).reset_index()
        simpan_csv(v, P_S13["variety"])
        tahap.selesai({"butir": len(d), "alpha_id2bbc": kes.set_index("arah").get("krippendorff_alpha_interval", {}).get("id2bbc"),
                       "alpha_bbc2id": kes.set_index("arah").get("krippendorff_alpha_interval", {}).get("bbc2id")})

---
# Bagian F. Paket hasil

## S14. Paket hasil untuk naskah

**Tujuan:** mengumpulkan semua tabel, gambar, dan catatan reproduktibilitas di satu folder `rilis/`.

**Yang dilakukan sel ini:**

1. Menyalin tabel dan gambar penting ke `rilis/`.
2. Membuat tabel LaTeX untuk tabel utama dan analisis arah asli.
3. Mencatat versi pustaka, GPU, dan konfigurasi lengkap ke `rilis/lingkungan.json`.
4. Membuat `rilis/manifest.json` berisi *hash* SHA-256 setiap berkas, supaya hasil dapat diverifikasi.
5. Mencetak status semua tahap.

**Keluaran yang diharapkan:** folder `rilis/` dan tabel status tahap. Status `dijalankan` berarti tahap dijalankan di sesi ini, `checkpoint` berarti diambil dari hasil sebelumnya, dan `belum` berarti belum pernah selesai.

In [ ]:
# S14 Paket hasil
import platform

berkas_rilis = [P_S03, P_S03_KAMI, P_S03_CONTOH, P_S04_SPLIT, P_S04_RINGKAS, P_S04_TUMPANG, P_S08_LOG,
                P_S10, P_S10_BEDA, P_S10_BENIH, P_S10_VAR, P_S11_UTAMA, P_S11_ARAH, P_S11_GENRE] + list(P_S13.values())
berkas_rilis += sorted(DIR_GAMBAR.glob("*.png"))
for p in berkas_rilis:
    if Path(p).exists():
        shutil.copy2(p, DIR_RILIS / Path(p).name)

for sumber, nama in [(P_S11_UTAMA, "tabel_utama.tex"), (P_S11_ARAH, "tabel_arah_asli.tex")]:
    if Path(sumber).exists():
        try:
            pd.read_csv(sumber).to_latex(DIR_RILIS / nama, index=False, na_rep="-", escape=True)
        except Exception as e:
            print(f"Tabel LaTeX {nama} dilewati:", e)

lingkungan = {"waktu": datetime.now(timezone.utc).isoformat(timespec="seconds"), "python": platform.python_version(),
              "gpu": info_gpu, "konfigurasi": KONFIGURASI}
for modul in ["torch", "transformers", "sacrebleu", "pandas", "numpy", "sklearn", "scipy"]:
    try:
        lingkungan[modul] = __import__(modul).__version__
    except Exception:
        lingkungan[modul] = "tidak ada"
tulis_atomik(DIR_RILIS / "lingkungan.json", json.dumps(lingkungan, indent=2, ensure_ascii=False, default=str))
manifest = {p.name: sha256_berkas(p) for p in sorted(DIR_RILIS.iterdir()) if p.is_file() and p.name != "manifest.json"}
tulis_atomik(DIR_RILIS / "manifest.json", json.dumps(manifest, indent=2))

status = []
for kode in ["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S09", "S10", "S11", "S12", "S13"]:
    meta = DIR_META / f"{kode}.json"
    status.append({"tahap": kode, "status": STATUS_TAHAP.get(kode, "checkpoint" if meta.exists() else "belum"),
                   "terakhir_selesai": json.loads(meta.read_text())["waktu"] if meta.exists() else ""})
tampilkan(pd.DataFrame(status), 20)
print(f"{len(manifest)} berkas di {DIR_RILIS}")

---
# Bagian G. Membaca hasil untuk naskah

| RQ | Berkas utama | Yang perlu diperhatikan |
|---|---|---|
| RQ1 mutu data | `s03_audit_varietas.csv`, laporan IAA tahap 1 (`data/iaa/` di GitHub) | Persentase AM korpus kita dibandingkan dengan NusaTranslation. Periksa `s03_contoh_am_korpus_kami.csv` secara manual. |
| RQ2 arah asli | `s11_arah_asli.csv`, `gambar/s11_arah_asli.png` | Tanda dan besar selisih `asli_toba` dikurangi `asli_indonesia`, apakah konsisten di semua sistem, dan IK-nya di `s10_skor.csv` (dimensi `arah_asli`). Bahas keterkaitannya dengan genre. |
| RQ3 model yang ada | `s11_tabel_utama.csv` | Apakah model *zero-shot* dan LLM melampaui *copy-source*, dan apakah MADLAD-400 benar-benar mengenal `<2bbc>` (`s07_ringkasan.json`). |
| RQ4 nilai data latih | `s10_perbandingan.csv`, `s10_rata_benih.csv` | Selisih `ft_kami+X` dikurangi `ft_X` pada set uji di luar domain. Selisih dengan IK yang tidak memuat nol menunjukkan manfaat yang nyata. |
| RQ5 contoh *prompt* | `s10_perbandingan.csv` (baris `llm_`) | `few_shot_retrieval` dibandingkan dengan `few_shot_acak` dan `zero_shot`. |
| RQ6 evaluasi | `s13_kesepakatan.csv`, `s13_korelasi_metrik.csv` | Alfa Krippendorff (≥ 0,667 sering dianggap batas bawah yang dapat diterima) dan Kendall tau-b, terutama di antara sistem `ft_`. |
| Varietas keluaran | `s10_varietas_keluaran.csv`, `gambar/s11_varietas_keluaran.png` | Apakah model yang dilatih NusaTranslation menghasilkan lebih banyak penanda AM dibanding model yang dilatih korpus kita. |

## Keterbatasan yang perlu ditulis di naskah

1. **Arah asli berkaitan dengan genre.** Selisih antar-arah asli belum tentu hanya disebabkan oleh *translationese*.
2. **Label arah berasal dari metadata sumber dan pemeriksaan peneliti**, bukan dari bukti penerjemahan per kalimat.
3. **Penanda varietas bersifat leksikal dan heuristik.** Angka AM adalah batas bawah.
4. **Set uji `kami` berukuran sekitar 300 kalimat**, dan subset asli Toba lebih kecil lagi. Interval kepercayaan perlu selalu dilaporkan.
5. ***Fine-tuning* hanya pada NLLB-200 600M**, dan LLM hanya diuji dengan satu templat *prompt*.
6. **Satu set penilai.** Dua penilai lebih baik daripada satu, tetapi pendapat dua orang belum tentu mewakili semua penutur Batak Toba.